# Retail Inventory Research

Inventory growth and subsequent operating performance in selected U.S.-listed merchandise retailers and consumer brands.

Final analysis/report version with partial filing review. Existing-project execution order: Modules 1-9, 12, 10, 11, 13. The reordered workflow uses the original Module 11 case-selection file already saved in this project. For a fresh project, create that file by following the original Modules 1-11 first.

Module 13 was executed against the saved reviewed-sample inputs. Earlier outputs retain the user's recorded runs; the entire combined notebook was not rerun during report assembly. See the final report and README for evidence coverage and limitations.


## Module 1: Retail Company Universe and SEC Identifiers

In [ ]:
%pip install requests pandas

In [24]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "work" / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [ ]:
import os
SEC_CONTACT = os.environ.get("SEC_CONTACT", "")
if not SEC_CONTACT:
    raise ValueError("Set the SEC_CONTACT environment variable to your contact email before acquiring SEC data.")
REFRESH = False

SEC_HEADERS = {
    "User-Agent": f"RetailInventoryResearch/0.1 (contact: {SEC_CONTACT})",
    "Accept": "application/json",
}
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers_exchange.json"

In [28]:
candidate_groups = {
    "Apparel, Footwear, and Off-Price Retail": "TJX ROST BURL LULU GAP URBN ANF BOOT AEO BKE CTRN JILL SFIX SHOE LE CURV ZUMZ DLTH TLYS AKA CATO LVLU DXLG".split(),
    "Department Stores": "M KSS DDS".split(),
    "Home Furnishings, Furniture, and Home Improvement": "HD LOW FND HVT WSM RH ARHS".split(),
    "Sporting Goods, Electronics, and Other Merchandise": "DKS ASO SPWH BBY GME FIVE".split(),
    "Beauty and Personal Care": "ULTA SBH BBWI".split(),
}
rows = []
for subgroup, tickers in candidate_groups.items():
    for ticker in tickers:
        rows.append({"ticker": ticker, "subgroup": subgroup})

candidates = pd.DataFrame(rows)
assert candidates["ticker"].is_unique, "Duplicate stock symbols found."
print("Candidate count:", len(candidates))
display(candidates.head())

Candidate count: 42


  ticker                                 subgroup
0    TJX  Apparel, Footwear, and Off-Price Retail
1   ROST  Apparel, Footwear, and Off-Price Retail
2   BURL  Apparel, Footwear, and Off-Price Retail
3   LULU  Apparel, Footwear, and Off-Price Retail
4    GAP  Apparel, Footwear, and Off-Price Retail

In [30]:
cache_path = RAW_DIR / "sec_company_tickers_exchange_snapshot.json"
if cache_path.exists() and not REFRESH:
    snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
    print("Source: local cache")
else:
    response = requests.get(SEC_TICKERS_URL, headers=SEC_HEADERS, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"SEC returned HTTP {response.status_code}. Stop and retry later.")
    snapshot = {
        "source_url": SEC_TICKERS_URL,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "payload": response.json(),
    }
    cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    print("Source: fresh SEC download")

payload = snapshot["payload"]
sec_tickers = pd.DataFrame(payload["data"], columns=payload["fields"])
display(sec_tickers.head())

Source: fresh SEC download


       cik            name ticker exchange
0  1045810     NVIDIA CORP   NVDA   Nasdaq
1   320193      Apple Inc.   AAPL   Nasdaq
2  1652044   Alphabet Inc.  GOOGL   Nasdaq
3   789019  MICROSOFT CORP   MSFT   Nasdaq
4  1018724  AMAZON COM INC   AMZN   Nasdaq

In [31]:
official = sec_tickers[sec_tickers["ticker"].isin(candidates["ticker"])]
official = official[["ticker", "cik", "name", "exchange"]]
official = official.rename(columns={"name": "sec_company_name"})
universe = candidates.merge(
    official, on="ticker", how="left", indicator=True, validate="one_to_one"
)
universe["cik"] = universe["cik"].astype("Int64")
universe["cik10"] = universe["cik"].astype("string").str.zfill(10)
status_mapping = {"both": "matched", "left_only": "needs_review", "right_only": "needs_review"}
universe["identity_status"] = universe["_merge"].map(status_mapping).astype("string")
universe["eligibility_status"] = "pending_business_and_data_audit"
universe["identity_snapshot_at_utc"] = snapshot["retrieved_at_utc"]
universe = universe.drop(columns="_merge")
display(universe.head())

  ticker                                 subgroup      cik  \
0    TJX  Apparel, Footwear, and Off-Price Retail   109198   
1   ROST  Apparel, Footwear, and Off-Price Retail   745732   
2   BURL  Apparel, Footwear, and Off-Price Retail  1579298   
3   LULU  Apparel, Footwear, and Off-Price Retail  1397187   
4    GAP  Apparel, Footwear, and Off-Price Retail    39911   

           sec_company_name exchange       cik10 identity_status  \
0    TJX COMPANIES INC /DE/     NYSE  0000109198         matched   
1         ROSS STORES, INC.   Nasdaq  0000745732         matched   
2   Burlington Stores, Inc.     NYSE  0001579298         matched   
3  lululemon athletica inc.   Nasdaq  0001397187         matched   
4                   GAP INC     NYSE  0000039911         matched   

                eligibility_status          identity_snapshot_at_utc  
0  pending_business_and_data_audit  2026-10-05T03:38:05.930604+00:00  
1  pending_business_and_data_audit  2026-10-05T03:38:05.930604+00:00  
2  pe

In [32]:
# Inspection of Result
matched = universe["identity_status"] == "matched"
print("Candidate count:", len(universe))
print("Matched count:", int(matched.sum()))
display(universe.loc[~matched, ["ticker", "identity_status"]])

Candidate count: 42
Matched count: 42


Empty DataFrame
Columns: [ticker, identity_status]
Index: []

In [33]:
universe_path = PROCESSED_DIR / "retail_candidate_universe.csv"
universe.to_csv(universe_path, index=False, encoding="utf-8-sig")
metadata = {
    "source_url": SEC_TICKERS_URL,
    "source_retrieved_at_utc": snapshot["retrieved_at_utc"],
    "candidate_count": len(universe),
    "business_and_data_eligibility_audited": False,
    "sampling_note": "Current listed-company candidates; historical exits are not yet included.",
    "grouping_note": "Research-defined categories pending annual-report review.",
}
metadata_path = PROCESSED_DIR / "retail_candidate_universe_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved universe:", universe_path)

Saved universe: [project root]/work/data/processed/retail_candidate_universe.csv


## Module 2: Download SEC Financial Data

In [ ]:
%pip install requests pandas

In [36]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR.mkdir(parents=True, exist_ok=True)

import os
SEC_CONTACT = os.environ.get("SEC_CONTACT", "")
if not SEC_CONTACT:
    raise ValueError("Set the SEC_CONTACT environment variable to your contact email before acquiring SEC data.")
REFRESH = False
SEC_HEADERS = {"User-Agent": f"RetailInventoryResearch/0.1 (contact: {SEC_CONTACT})"}
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [37]:
universe_path = PROCESSED_DIR / "retail_candidate_universe.csv"
universe = pd.read_csv(universe_path, dtype={"ticker": "string", "cik10": "string"})
PILOT_TICKERS = ["AEO", "ANF", "URBN"]
pilot = universe.loc[universe["ticker"].isin(PILOT_TICKERS)].copy()
if len(pilot) != 3 or pilot["cik10"].isna().any():
    raise ValueError("The pilot companies or CIKs are missing. Check Module 1.")
display(pilot[["ticker", "sec_company_name", "cik10"]])

  ticker               sec_company_name       cik10
5   URBN           URBAN OUTFITTERS INC  0000912615
6    ANF    ABERCROMBIE & FITCH CO /DE/  0001018840
8    AEO  AMERICAN EAGLE OUTFITTERS INC  0000919012

In [38]:
company_facts = {}
for index, row in pilot.iterrows():
    ticker = row["ticker"]
    cik10 = row["cik10"]
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
    cache_path = FACTS_DIR / f"CIK{cik10}_companyfacts.json"

    if cache_path.exists() and not REFRESH:
        snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
        origin = "cache"
    else:
        time.sleep(1)
        response = requests.get(url, headers=SEC_HEADERS, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(f"{ticker}: HTTP {response.status_code}. Stop and retry later.")
        snapshot = {
            "source_url": url,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "payload": response.json(),
        }
        origin = "download"

    if int(snapshot["payload"]["cik"]) != int(cik10):
        raise ValueError(f"{ticker}: CIK mismatch in the snapshot.")
    if origin == "download":
        cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    company_facts[ticker] = snapshot["payload"]
    print(ticker, origin, snapshot["retrieved_at_utc"])

URBN download 2026-10-05T03:51:12.952266+00:00
ANF download 2026-10-05T03:51:14.951742+00:00
AEO download 2026-10-05T03:51:16.742098+00:00


In [39]:
aeo = company_facts["AEO"]
us_gaap = aeo["facts"].get("us-gaap", {})
print("Company:", aeo["entityName"])
print("US GAAP tag count:", len(us_gaap))
print("Example tags:", list(us_gaap.keys())[:10])

Company: American Eagle Outfitters, Inc.
US GAAP tag count: 481
Example tags: ['AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccruedIncomeTaxesNoncurrent', 'AccruedLiabilitiesForUnredeeemedGiftCards', 'AccruedRentCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossBeforeTax1', 'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossOtherThanTemporaryImpairmentNotCreditLossNetOfTaxDebtSecurities', 'AdditionalPaidInCapitalCommonStock']


In [40]:
tag_candidates = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues", "SalesRevenueGoodsNet"],
    "cost_of_sales": ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"],
    "inventory": ["InventoryNet"],
    "total_assets": ["Assets"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
}
rows = []
for ticker, data in company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, possible_tags in tag_candidates.items():
        found = False
        for tag in possible_tags:
            if tag in tags:
                records = tags[tag]["units"].get("USD", [])
                rows.append({"ticker": ticker, "metric": metric, "tag": tag, "record_count_usd": len(records)})
                found = True
        if not found:
            rows.append({"ticker": ticker, "metric": metric, "tag": None, "record_count_usd": 0})

tag_summary = pd.DataFrame(rows)
display(tag_summary)

   ticker               metric  \
0    URBN              revenue   
1    URBN              revenue   
2    URBN        cost_of_sales   
3    URBN        cost_of_sales   
4    URBN            inventory   
5    URBN         total_assets   
6    URBN  operating_cash_flow   
7    URBN  operating_cash_flow   
8     ANF              revenue   
9     ANF              revenue   
10    ANF        cost_of_sales   
11    ANF        cost_of_sales   
12    ANF            inventory   
13    ANF         total_assets   
14    ANF  operating_cash_flow   
15    AEO              revenue   
16    AEO              revenue   
17    AEO        cost_of_sales   
18    AEO        cost_of_sales   
19    AEO            inventory   
20    AEO         total_assets   
21    AEO  operating_cash_flow   
22    AEO  operating_cash_flow   

                                                  tag  record_count_usd  
0   RevenueFromContractWithCustomerExcludingAssess...               110  
1                                  

In [41]:
inventory_data = us_gaap.get("InventoryNet", {}).get("units", {}).get("USD", [])
inventory = pd.DataFrame(inventory_data)
if inventory.empty:
    print("No USD InventoryNet records found for AEO. Review the annual report.")
else:
    inventory = inventory.sort_values(["filed", "end"])
    display(inventory[["end", "val", "form", "filed", "accn"]].tail(8))

summary_path = PROCESSED_DIR / "pilot_financial_tag_summary.csv"
tag_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("Saved tag summary:", summary_path)
print("Raw financial snapshots:", FACTS_DIR)

            end        val  form       filed                  accn
168  2025-02-01  636655000  10-K  2026-03-30  0001193125-26-132097
174  2026-01-31  701966000  10-K  2026-03-30  0001193125-26-132097
170  2025-05-03  645062000  10-Q  2026-06-03  0001193125-26-255712
175  2026-01-31  701966000  10-Q  2026-06-03  0001193125-26-255712
177  2026-05-02  816666000  10-Q  2026-06-03  0001193125-26-255712
172  2025-08-02  718337000  10-Q  2026-09-10  0001193125-26-387856
176  2026-01-31  701966000  10-Q  2026-09-10  0001193125-26-387856
178  2026-08-01  817910000  10-Q  2026-09-10  0001193125-26-387856

Saved tag summary: [project root]/work/data/processed/pilot_financial_tag_summary.csv
Raw financial snapshots: [project root]/work/data/raw/companyfacts


## Module 3: Build an Annual Financial Panel

In [ ]:
%pip install pandas

In [43]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [44]:
universe = pd.read_csv(PROCESSED_DIR / "retail_candidate_universe.csv", dtype={"cik10": "string"})
pilot = universe.loc[universe["ticker"].isin(["AEO", "ANF", "URBN"])]
if len(pilot) != 3 or pilot["cik10"].isna().any():
    raise ValueError("The pilot companies or CIKs are missing. Check Module 1.")

company_facts = {}
snapshot_times = {}
for index, row in pilot.iterrows():
    cache_path = FACTS_DIR / f"CIK{row['cik10']}_companyfacts.json"
    snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
    if int(snapshot["payload"]["cik"]) != int(row["cik10"]):
        raise ValueError(f"{row['ticker']}: CIK mismatch in the cache.")
    company_facts[row["ticker"]] = snapshot["payload"]
    snapshot_times[row["ticker"]] = snapshot["retrieved_at_utc"]
    print(row["ticker"], "loaded", snapshot["retrieved_at_utc"])

URBN loaded 2026-10-05T03:51:12.952266+00:00
ANF loaded 2026-10-05T03:51:14.951742+00:00
AEO loaded 2026-10-05T03:51:16.742098+00:00


In [50]:
tag_candidates = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues", "SalesRevenueGoodsNet"],
    "cost_of_sales": [
        "CostOfRevenue",
        "CostOfGoodsAndServicesSold",
        "CostOfGoodsSold",
        "CostOfGoodsSoldExcludingDepreciationDepletionAndAmortization",
        "CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization",
    ],
    "gross_profit": ["GrossProfit"],
    "inventory": ["InventoryNet"],
    "total_assets": ["Assets"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
}
rows = []
for ticker, data in company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, possible_tags in tag_candidates.items():
        for priority, tag in enumerate(possible_tags):
            records = tags.get(tag, {}).get("units", {}).get("USD", [])
            for fact in records:
                rows.append({
                    "ticker": ticker, "metric": metric, "tag": tag, "priority": priority,
                    "start": fact.get("start"), "end": fact["end"], "value": fact["val"],
                    "form": fact["form"], "filed": fact["filed"], "accn": fact["accn"],
                })

facts = pd.DataFrame(rows)
if facts.empty:
    raise ValueError("No candidate USD records found. Review Module 2 and the annual reports.")
for column in ["start", "end", "filed"]:
    facts[column] = pd.to_datetime(facts[column])
facts = facts.drop_duplicates()
display(facts.head())

  ticker   metric                                                tag  \
0   URBN  revenue  RevenueFromContractWithCustomerExcludingAssess...   
1   URBN  revenue  RevenueFromContractWithCustomerExcludingAssess...   
2   URBN  revenue  RevenueFromContractWithCustomerExcludingAssess...   
3   URBN  revenue  RevenueFromContractWithCustomerExcludingAssess...   
4   URBN  revenue  RevenueFromContractWithCustomerExcludingAssess...   

   priority      start        end       value  form      filed  \
0         0 2016-02-01 2017-01-31  3545794000  10-K 2019-04-01   
1         0 2017-02-01 2017-04-30   761190000  10-Q 2018-06-11   
2         0 2017-02-01 2017-07-31  1634121000  10-Q 2018-09-10   
3         0 2017-05-01 2017-07-31   872931000  10-Q 2018-09-10   
4         0 2017-02-01 2017-10-31  2526895000  10-Q 2018-12-10   

                   accn  
0  0001564590-19-010433  
1  0001564590-18-015394  
2  0001564590-18-022875  
3  0001564590-18-022875  
4  0001564590-18-030731  

In [51]:
facts["period_days"] = (facts["end"] - facts["start"]).dt.days + 1
annual_revenue = facts.loc[
    (facts["metric"] == "revenue")
    & facts["form"].isin(["10-K", "10-K/A"])
    & facts["period_days"].between(350, 380)
].copy()
anchors = annual_revenue.sort_values(["ticker", "end", "filed", "accn", "priority"])
anchors = anchors.drop_duplicates(["ticker", "end"], keep="first")
if anchors.empty:
    raise ValueError("No full-year revenue periods found. Review the revenue tags.")
display(anchors[["ticker", "start", "end", "period_days", "filed", "accn", "tag"]].tail(10))

    ticker      start        end  period_days      filed  \
204   URBN 2016-02-01 2017-01-31        366.0 2017-04-03   
211   URBN 2017-02-01 2018-01-31        365.0 2018-04-02   
18    URBN 2018-02-01 2019-01-31        365.0 2019-04-01   
31    URBN 2019-02-01 2020-01-31        365.0 2020-03-31   
44    URBN 2020-02-01 2021-01-31        366.0 2021-04-01   
57    URBN 2021-02-01 2022-01-31        365.0 2022-04-01   
70    URBN 2022-02-01 2023-01-31        365.0 2023-04-03   
83    URBN 2023-02-01 2024-01-31        365.0 2024-04-01   
96    URBN 2024-02-01 2025-01-31        366.0 2025-04-01   
106   URBN 2025-02-01 2026-01-31        365.0 2026-04-01   

                     accn                                                tag  
204  0001193125-17-108506                                    SalesRevenueNet  
211  0001564590-18-007408                                    SalesRevenueNet  
18   0001564590-19-010433  RevenueFromContractWithCustomerExcludingAssess...  
31   0001564590-20-0143

In [52]:
annual_rows = []
source_rows = []
for index, year in anchors.iterrows():
    annual_row = {
        "ticker": year["ticker"], "period_start": year["start"], "period_end": year["end"],
        "period_days": year["period_days"], "filed": year["filed"], "accn": year["accn"],
    }
    for metric in tag_candidates:
        matches = facts.loc[
            (facts["ticker"] == year["ticker"]) & (facts["metric"] == metric)
            & (facts["end"] == year["end"]) & (facts["accn"] == year["accn"])
        ]
        if metric not in ["inventory", "total_assets"]:
            matches = matches.loc[matches["start"] == year["start"]]
        value = None
        tag = None
        status = "missing"
        if not matches.empty:
            matches = matches.sort_values("priority")
            tag = matches.iloc[0]["tag"]
            chosen = matches.loc[matches["tag"] == tag]
            if chosen["value"].nunique() == 1:
                value = chosen.iloc[0]["value"]
                status = "selected_pending_review"
            else:
                status = "ambiguous"
        annual_row[metric] = value
        source_rows.append({
            "ticker": year["ticker"], "period_end": year["end"], "metric": metric,
            "tag": tag, "value": value, "status": status, "accn": year["accn"],
        })
    annual_rows.append(annual_row)

annual_panel = pd.DataFrame(annual_rows).sort_values(["ticker", "period_end"])
field_sources = pd.DataFrame(source_rows)
display(annual_panel.head())

  ticker period_start period_end  period_days      filed  \
0    AEO   2008-02-03 2009-01-31        364.0 2011-03-11   
1    AEO   2009-02-01 2010-01-30        364.0 2011-03-11   
2    AEO   2010-01-31 2011-01-29        364.0 2011-03-11   
3    AEO   2011-01-30 2012-01-28        364.0 2012-03-15   
4    AEO   2012-01-29 2013-02-02        371.0 2013-03-12   

                   accn     revenue  cost_of_sales  gross_profit    inventory  \
0  0000950123-11-024788  2948679000            NaN  1.197186e+09          NaN   
1  0000950123-11-024788  2940269000            NaN  1.173430e+09  326454000.0   
2  0000950123-11-024788  2967559000            NaN  1.170959e+09  301208000.0   
3  0001193125-12-117468  3159818000            NaN  1.128341e+09  378426000.0   
4  0001193125-13-102293  3475802000            NaN  1.390322e+09  332452000.0   

   total_assets  operating_cash_flow  
0           NaN            345112000  
1  2.138148e+09            400326000  
2  1.879998e+09            40259400

In [ ]:
# Keep a record of how each derived value is calculated.
if "derivation" not in field_sources.columns:
    field_sources["derivation"] = ""

for index, row in annual_panel.iterrows():

    # Recover AEO cost of sales for the reviewed annual periods.
    if (
        row["ticker"] == "AEO"
        and pd.Timestamp("2009-01-31") <= row["period_end"] <= pd.Timestamp("2015-01-31")
        and pd.isna(row["cost_of_sales"])
    ):
        metric = "cost_of_sales"
        value = row["revenue"] - row["gross_profit"]
        formula = "revenue - gross_profit"

    # Calculate ANF gross profit using its disclosed cost definition.
    elif (
        row["ticker"] == "ANF"
        and row["period_end"] in [
            pd.Timestamp("2025-02-01"),
            pd.Timestamp("2026-01-31"),
        ]
        and pd.isna(row["gross_profit"])
    ):
        metric = "gross_profit"
        value = row["revenue"] - row["cost_of_sales"]
        formula = "revenue - cost_of_sales; cost excludes depreciation and amortization"

    else:
        continue

    # Leave the result missing if either input is unavailable.
    if pd.isna(value):
        continue

    source_match = (
        (field_sources["ticker"] == row["ticker"])
        & (field_sources["period_end"] == row["period_end"])
        & (field_sources["accn"] == row["accn"])
        & (field_sources["metric"] == metric)
    )

    # Preserve records that require a separate conflict review.
    if not field_sources.loc[source_match, "status"].eq("missing").all():
        continue

    annual_panel.loc[index, metric] = value
    field_sources.loc[source_match, "value"] = value
    field_sources.loc[source_match, "status"] = "derived_pending_review"
    field_sources.loc[source_match, "derivation"] = formula

In [59]:
display(annual_panel.loc[annual_panel["ticker"] == "AEO"].tail(5))

metrics = list(tag_candidates.keys())
display(annual_panel[metrics].isna().sum().to_frame("Missing Rows"))

# Show unresolved records.
display(
    field_sources.loc[
        field_sources["status"].isin(["missing", "ambiguous"])
    ]
)

# Show values calculated from other financial fields.
display(
    field_sources.loc[
        field_sources["status"] == "derived_pending_review"
    ]
)

print("Annual period rows:", len(annual_panel))

   ticker period_start period_end  period_days      filed  \
13    AEO   2021-01-31 2022-01-29        364.0 2022-03-14   
14    AEO   2022-01-30 2023-01-28        364.0 2023-03-13   
15    AEO   2023-01-29 2024-02-03        371.0 2024-03-15   
16    AEO   2024-02-04 2025-02-01        364.0 2025-03-20   
17    AEO   2025-02-02 2026-01-31        364.0 2026-03-30   

                    accn     revenue  cost_of_sales  gross_profit  \
13  0000950170-22-003587  5010785000   3.018995e+09  1.991790e+09   
14  0000950170-23-007604  4989833000   3.244585e+09  1.745248e+09   
15  0000950170-24-032294  5261770000   3.237192e+09  2.024578e+09   
16  0000950170-25-042746  5328652000   3.239719e+09  2.088933e+09   
17  0001193125-26-132097  5547236000   3.521915e+09  2.025321e+09   

      inventory  total_assets  operating_cash_flow  
13  553458000.0  3.786643e+09            303671000  
14  585083000.0  3.420956e+09            406296000  
15  640662000.0  3.557909e+09            580710000  
16  63

                     Missing Rows
revenue                         0
cost_of_sales                   0
gross_profit                    0
inventory                       3
total_assets                    3
operating_cash_flow             0

    ticker period_end        metric  tag  value   status  \
3      AEO 2009-01-31     inventory  NaN    NaN  missing   
4      AEO 2009-01-31  total_assets  NaN    NaN  missing   
111    ANF 2009-01-31     inventory  NaN    NaN  missing   
112    ANF 2009-01-31  total_assets  NaN    NaN  missing   
219   URBN 2009-01-31     inventory  NaN    NaN  missing   
220   URBN 2009-01-31  total_assets  NaN    NaN  missing   

                     accn derivation  
3    0000950123-11-024788             
4    0000950123-11-024788             
111  0000950123-11-030172             
112  0000950123-11-030172             
219  0001193125-11-085784             
220  0001193125-11-085784             

    ticker period_end         metric  tag         value  \
1      AEO 2009-01-31  cost_of_sales  NaN  1.751493e+09   
7      AEO 2010-01-30  cost_of_sales  NaN  1.766839e+09   
13     AEO 2011-01-29  cost_of_sales  NaN  1.796600e+09   
19     AEO 2012-01-28  cost_of_sales  NaN  2.031477e+09   
25     AEO 2013-02-02  cost_of_sales  NaN  2.085480e+09   
31     AEO 2014-02-01  cost_of_sales  NaN  2.191803e+09   
37     AEO 2015-01-31  cost_of_sales  NaN  2.128193e+09   
206    ANF 2025-02-01   gross_profit  NaN  3.174661e+09   
212    ANF 2026-01-31   gross_profit  NaN  3.237408e+09   

                     status                  accn  \
1    derived_pending_review  0000950123-11-024788   
7    derived_pending_review  0000950123-11-024788   
13   derived_pending_review  0000950123-11-024788   
19   derived_pending_review  0001193125-12-117468   
25   derived_pending_review  0001193125-13-102293   
31   derived_pending_review  0001193125-14-098050   
37   derived_pending_review  000119312

Annual period rows: 54


In [60]:
# Set the output file paths.
panel_path = PROCESSED_DIR / "pilot_annual_panel.csv"
sources_path = PROCESSED_DIR / "pilot_annual_field_sources.csv"
metadata_path = PROCESSED_DIR / "pilot_annual_panel_metadata.json"

# Save the annual financial panel.
annual_panel.to_csv(
    panel_path,
    index=False,
    date_format="%Y-%m-%d",
)

# Save field values, source tags, statuses, and derivation formulas.
field_sources.to_csv(
    sources_path,
    index=False,
    date_format="%Y-%m-%d",
)

# Record the data sources and processing rules.
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "snapshot_times": snapshot_times,
    "unit": "USD",
    "annual_period_days": [350, 380],
    "filing_rule": (
        "Earliest available full-year revenue 10-K/10-K/A per period end; "
        "other metrics use the same accession."
    ),
    "tag_candidates_in_priority_order": tag_candidates,
    "derivation_rules": {
        "AEO": {
            "metric": "cost_of_sales",
            "period_end_from": "2009-01-31",
            "period_end_to": "2015-01-31",
            "formula": "revenue - gross_profit",
            "condition": "Apply only when cost of sales is missing.",
        },
        "ANF": {
            "metric": "gross_profit",
            "period_ends": ["2025-02-01", "2026-01-31"],
            "formula": "revenue - cost_of_sales",
            "condition": "Apply only when gross profit is missing.",
            "cost_scope": "Excludes depreciation and amortization.",
        },
    },
    "derived_value_status": "derived_pending_review",
    "missing_value_policy": "Retain unresolved missing values without imputation.",
    "accounting_scope_verified": False,
}

# Save the processing metadata.
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Saved annual panel:", panel_path)
print("Saved field sources:", sources_path)
print("Saved metadata:", metadata_path)

Saved annual panel: [project root]/work/data/processed/pilot_annual_panel.csv
Saved field sources: [project root]/work/data/processed/pilot_annual_field_sources.csv
Saved metadata: [project root]/work/data/processed/pilot_annual_panel_metadata.json


## Module 4: Review Accounting Scope and Build a Pilot Research Sample

In [61]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [62]:
panel = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_panel.csv",
    parse_dates=["period_start", "period_end", "filed"],
)
sources = pd.read_csv(PROCESSED_DIR / "pilot_annual_field_sources.csv")
module3_metadata = json.loads(
    (PROCESSED_DIR / "pilot_annual_panel_metadata.json").read_text(encoding="utf-8")
)
panel = panel.sort_values(["ticker", "period_end"]).reset_index(drop=True)
metrics = ["revenue", "cost_of_sales", "gross_profit", "inventory", "total_assets", "operating_cash_flow"]
print("Annual rows loaded:", len(panel))
display(sources["status"].value_counts().to_frame("Field Count"))

Annual rows loaded: 54


                         Field Count
status                              
selected_pending_review          309
derived_pending_review             9
missing                            6

In [63]:
scope_notes = pd.DataFrame([
    {"ticker": "AEO", "cost_scope": "Includes certain buying, occupancy, and warehousing expenses.",
     "business_scope": "Historical 77kids discontinued-operation reclassification requires review.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm"},
    {"ticker": "ANF", "cost_scope": "Excludes depreciation and amortization; expense classifications differ across retailers.",
     "business_scope": "Consolidated company data; historical scope review is pending.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/1018840/000101884026000012/anf-20260131.htm"},
    {"ticker": "URBN", "cost_scope": "Reported gross profit can also deduct separately presented store charges.",
     "business_scope": "Includes retail, wholesale, and subscription operations.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf"},
])
scope_notes["historical_scope_review"] = "pending"
display(scope_notes)

  ticker                                         cost_scope  \
0    AEO  Includes certain buying, occupancy, and wareho...   
1    ANF  Excludes depreciation and amortization; expens...   
2   URBN  Reported gross profit can also deduct separate...   

                                      business_scope  \
0  Historical 77kids discontinued-operation recla...   
1  Consolidated company data; historical scope re...   
2  Includes retail, wholesale, and subscription o...   

                                          source_url historical_scope_review  
0  https://www.sec.gov/Archives/edgar/data/919012...                 pending  
1  https://www.sec.gov/Archives/edgar/data/101884...                 pending  
2  https://www.sec.gov/Archives/edgar/data/912615...                 pending  

In [64]:
audit = panel.copy()
audit["amounts_available"] = audit[metrics].notna().all(axis=1)
audit["basic_values_ok"] = (
    (audit["revenue"] > 0) & (audit["total_assets"] > 0)
    & (audit["inventory"] >= 0) & (audit["cost_of_sales"] >= 0)
)
audit["row_usable"] = audit["amounts_available"] & audit["basic_values_ok"]
audit["gross_profit_bridge_gap"] = audit["revenue"] - audit["cost_of_sales"] - audit["gross_profit"]
audit["known_scope_break"] = (
    (audit["ticker"] == "AEO") & (audit["period_end"] == pd.Timestamp("2013-02-02"))
)
display(audit.groupby("ticker")[["amounts_available", "row_usable"]].sum())
display(audit.loc[~audit["row_usable"], ["ticker", "period_end"] + metrics])
display(audit.loc[
    audit["gross_profit_bridge_gap"].abs() > 1,
    ["ticker", "period_end", "gross_profit_bridge_gap"],
])

        amounts_available  row_usable
ticker                               
AEO                    17          17
ANF                    17          17
URBN                   17          17

   ticker period_end     revenue  cost_of_sales  gross_profit  inventory  \
0     AEO 2009-01-31  2948679000   1.751493e+09  1.197186e+09        NaN   
18    ANF 2009-01-31  3484058000   1.152963e+09  2.331095e+09        NaN   
36   URBN 2009-01-31  1834618000   1.121140e+09  7.134780e+08        NaN   

    total_assets  operating_cash_flow  
0            NaN            345112000  
18           NaN            491031000  
36           NaN            251570000  

   ticker period_end  gross_profit_bridge_gap
47   URBN 2020-01-31               14611000.0
48   URBN 2021-01-31               15496000.0
50   URBN 2023-01-31                6417000.0
51   URBN 2024-01-31               11875000.0
52   URBN 2025-01-31                4601000.0
53   URBN 2026-01-31                1989000.0

In [65]:
audit["gross_margin"] = audit["gross_profit"] / audit["revenue"]
audit["ocf_margin"] = audit["operating_cash_flow"] / audit["revenue"]
audit["inventory_to_assets"] = audit["inventory"] / audit["total_assets"]
ratio_columns = ["gross_margin", "ocf_margin", "inventory_to_assets"]
audit.loc[~audit["row_usable"], ratio_columns] = float("nan")
display(audit[["ticker", "period_end"] + ratio_columns].tail(6))

   ticker period_end  gross_margin  ocf_margin  inventory_to_assets
48   URBN 2021-01-31      0.249846    0.082851             0.109865
49   URBN 2022-01-31      0.328430    0.078993             0.150263
50   URBN 2023-01-31      0.297632    0.029765             0.159523
51   URBN 2024-01-31      0.332879    0.098853             0.133839
52   URBN 2025-01-31      0.347106    0.090590             0.137437
53   URBN 2026-01-31      0.359711    0.093294             0.139976

In [66]:
for column in ["revenue", "inventory", "gross_margin", "ocf_margin", "period_end", "period_days"]:
    audit["previous_" + column] = audit.groupby("ticker")[column].shift(1)
previous_row_usable = audit.groupby("ticker")["row_usable"].shift(1).eq(True)
year_gap = (audit["period_end"] - audit["previous_period_end"]).dt.days
audit["growth_usable"] = (
    audit["row_usable"] & previous_row_usable & year_gap.between(350, 380)
    & (audit["previous_revenue"] > 0) & (audit["previous_inventory"] > 0)
    & ~audit["known_scope_break"]
)
audit["period_length_changed"] = (
    (audit["period_days"] - audit["previous_period_days"]).abs() > 3
)
audit["revenue_growth"] = audit["revenue"] / audit["previous_revenue"] - 1
audit["inventory_growth"] = audit["inventory"] / audit["previous_inventory"] - 1
audit["inventory_sales_gap"] = audit["inventory_growth"] - audit["revenue_growth"]
audit["gross_margin_change"] = audit["gross_margin"] - audit["previous_gross_margin"]
audit["ocf_margin_change"] = audit["ocf_margin"] - audit["previous_ocf_margin"]
change_columns = ["revenue_growth", "inventory_growth", "inventory_sales_gap", "gross_margin_change", "ocf_margin_change"]
audit.loc[~audit["growth_usable"], change_columns] = float("nan")
display(audit[["ticker", "period_end", "growth_usable", "period_length_changed"] + change_columns].tail(6))

   ticker period_end  growth_usable  period_length_changed  revenue_growth  \
48   URBN 2021-01-31           True                  False       -0.134053   
49   URBN 2022-01-31           True                  False        0.318578   
50   URBN 2023-01-31           True                  False        0.054186   
51   URBN 2024-01-31           True                  False        0.074656   
52   URBN 2025-01-31           True                  False        0.077122   
53   URBN 2026-01-31           True                  False        0.110745   

    inventory_growth  inventory_sales_gap  gross_margin_change  \
48         -0.048631             0.085422            -0.061372   
49          0.462199             0.143621             0.078584   
50          0.031264            -0.022923            -0.030798   
51         -0.063434            -0.138090             0.035247   
52          0.128860             0.051737             0.014227   
53          0.128471             0.017725             0.0

In [67]:
research_panel = audit.loc[audit["row_usable"]].copy()
inventory_features = audit.loc[audit["growth_usable"]].copy()
print("Complete pilot annual rows:", len(research_panel))
print("Eligible pilot growth rows:", len(inventory_features))
display(inventory_features.groupby("ticker").size().to_frame("Growth Rows"))
display(inventory_features.loc[
    inventory_features["ticker"] == "AEO",
    ["ticker", "period_end", "period_length_changed"] + change_columns,
].tail(5))

Complete pilot annual rows: 51
Eligible pilot growth rows: 47


        Growth Rows
ticker             
AEO              15
ANF              16
URBN             16

   ticker period_end  period_length_changed  revenue_growth  inventory_growth  \
13    AEO 2022-01-29                  False        0.332970          0.365063   
14    AEO 2023-01-28                  False       -0.004181          0.057141   
15    AEO 2024-02-03                   True        0.054498          0.094993   
16    AEO 2025-02-01                   True        0.012711         -0.006254   
17    AEO 2026-01-31                  False        0.041021          0.102585   

    inventory_sales_gap  gross_margin_change  ocf_margin_change  
13             0.032093             0.092070           0.006735  
14             0.061322            -0.047740           0.020821  
15             0.040495             0.035010           0.028939  
16            -0.018965             0.007248          -0.020886  
17             0.061564            -0.026915          -0.007242  

In [68]:
outputs = {
    "pilot_annual_audit.csv": audit,
    "pilot_company_scope_notes.csv": scope_notes,
    "pilot_research_panel.csv": research_panel,
    "pilot_inventory_features.csv": inventory_features,
}
for filename, data in outputs.items():
    data.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
    print("Saved:", filename)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module3_metadata": module3_metadata,
    "unit": "USD; ratios and changes stored as decimals",
    "complete_annual_rows": len(research_panel),
    "eligible_growth_rows": len(inventory_features),
    "known_scope_break": "Exclude AEO growth comparison ending 2013-02-02 due to discontinued-operation reclassification.",
    "growth_rule": "Consecutive complete annual rows, 350-380 day end-date gap, positive prior revenue and inventory, excluding known scope breaks.",
    "period_length_policy": "Flag material length changes; do not normalize revenue automatically.",
    "accounting_scope_verified": False,
    "prediction_sample_ready": False,
}
(PROCESSED_DIR / "pilot_module4_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_module4_metadata.json")

Saved: pilot_annual_audit.csv
Saved: pilot_company_scope_notes.csv
Saved: pilot_research_panel.csv
Saved: pilot_inventory_features.csv
Saved: pilot_module4_metadata.json


## Module 5: Pair Inventory Features with Next-Year Outcomes

In [69]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [70]:
annual = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_audit.csv",
    parse_dates=["period_start", "period_end", "filed"],
)
features = pd.read_csv(
    PROCESSED_DIR / "pilot_inventory_features.csv",
    parse_dates=["period_start", "period_end", "filed", "previous_period_end"],
)
module4_metadata = json.loads(
    (PROCESSED_DIR / "pilot_module4_metadata.json").read_text(encoding="utf-8")
)
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
features = features.sort_values(["ticker", "period_end"]).reset_index(drop=True)
print("Annual lookup rows:", len(annual))
print("Current feature rows:", len(features))

Annual lookup rows: 54
Current feature rows: 47


In [71]:
future = annual[["ticker", "period_end"]].copy()
for column in [
    "period_start", "period_end", "period_days", "filed", "accn",
    "gross_margin", "ocf_margin", "row_usable", "known_scope_break",
]:
    future["next_" + column] = annual.groupby("ticker")[column].shift(-1)

paired = features.merge(
    future, on=["ticker", "period_end"], how="left", validate="one_to_one"
)
display(paired[[
    "ticker", "period_end", "next_period_start", "next_period_end", "next_known_scope_break",
]].head(8))

  ticker period_end next_period_start next_period_end next_known_scope_break
0    AEO 2011-01-29        2011-01-30      2012-01-28                  False
1    AEO 2012-01-28        2012-01-29      2013-02-02                   True
2    AEO 2014-02-01        2014-02-02      2015-01-31                  False
3    AEO 2015-01-31        2015-02-01      2016-01-30                  False
4    AEO 2016-01-30        2016-01-31      2017-01-28                  False
5    AEO 2017-01-28        2017-01-29      2018-02-03                  False
6    AEO 2018-02-03        2018-02-04      2019-02-02                  False
7    AEO 2019-02-02        2019-02-03      2020-02-01                  False

In [72]:
end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = end_gap.between(350, 380) & start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (
    (paired["signal_available_date"] >= paired["period_end"])
    & (paired["signal_available_date"] < paired["next_period_end"])
)
paired["available_before_target_start"] = (
    paired["signal_available_date"] < paired["next_period_start"]
)
paired["days_after_target_start"] = (
    paired["signal_available_date"] - paired["next_period_start"]
).dt.days
paired["pair_usable"] = (
    paired["next_period_end"].notna() & paired["consecutive_next_year"]
    & paired["next_row_usable"].eq(True)
    & paired["next_known_scope_break"].eq(False) & paired["filed_time_ok"]
)
display(paired.loc[~paired["pair_usable"], [
    "ticker", "period_end", "next_period_end", "next_row_usable",
    "next_known_scope_break", "consecutive_next_year", "filed_time_ok",
]])

   ticker period_end next_period_end next_row_usable next_known_scope_break  \
1     AEO 2012-01-28      2013-02-02            True                   True   
14    AEO 2026-01-31             NaT             NaN                    NaN   
30    ANF 2026-01-31             NaT             NaN                    NaN   
46   URBN 2026-01-31             NaT             NaN                    NaN   

    consecutive_next_year  filed_time_ok  
1                    True           True  
14                  False          False  
30                  False          False  
46                  False          False  

In [ ]:
paired["next_gross_margin_change"] = (
    paired["next_gross_margin"] - paired["gross_margin"]
)
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
outcome_columns = ["next_gross_margin_change", "next_ocf_margin_change"]
paired.loc[~paired["pair_usable"], outcome_columns] = float("nan")

for outcome, label in [
    ("next_gross_margin_change", "gross_margin_declined"),
    ("next_ocf_margin_change", "ocf_margin_declined"),
]:
    paired[label] = (paired[outcome] < 0).astype("boolean")
    paired.loc[~paired["pair_usable"], label] = pd.NA
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"] > 0

In [75]:
paired["pair_period_length_changed"] = (
    paired["period_length_changed"]
    | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
)
valid = paired["pair_usable"]
print("Usable next-year pairs:", int(valid.sum()))
print("Available before target year starts:", int(
    paired.loc[valid, "available_before_target_start"].sum()
))
display(paired.loc[valid, "days_after_target_start"].agg(["min", "max"]))
display(paired.loc[valid, [
    "ticker", "period_end", "signal_available_date", "next_period_start",
    "next_period_end", "days_after_target_start", "pair_period_length_changed",
]].tail(6))

Usable next-year pairs: 43
Available before target year starts: 0


min    38.0
max    61.0
Name: days_after_target_start, dtype: float64

   ticker period_end signal_available_date next_period_start next_period_end  \
40   URBN 2020-01-31            2020-03-31        2020-02-01      2021-01-31   
41   URBN 2021-01-31            2021-04-01        2021-02-01      2022-01-31   
42   URBN 2022-01-31            2022-04-01        2022-02-01      2023-01-31   
43   URBN 2023-01-31            2023-04-03        2023-02-01      2024-01-31   
44   URBN 2024-01-31            2024-04-01        2024-02-01      2025-01-31   
45   URBN 2025-01-31            2025-04-01        2025-02-01      2026-01-31   

    days_after_target_start  pair_period_length_changed  
40                     59.0                       False  
41                     59.0                       False  
42                     59.0                       False  
43                     61.0                       False  
44                     60.0                       False  
45                     59.0                       False  

In [76]:
next_year_sample = paired.loc[paired["pair_usable"]].copy()
research_columns = [
    "ticker", "period_end", "inventory_sales_gap", "next_period_end",
    "next_gross_margin_change", "next_ocf_margin_change",
    "gross_margin_declined", "ocf_margin_declined",
]
print("Pilot paired rows:", len(next_year_sample))
display(next_year_sample.groupby("ticker").size().to_frame("Paired Rows"))
display(next_year_sample[research_columns].tail(8))

Pilot paired rows: 43


        Paired Rows
ticker             
AEO              13
ANF              15
URBN             15

   ticker period_end  inventory_sales_gap next_period_end  \
38   URBN 2018-01-31             0.018015      2019-01-31   
39   URBN 2019-01-31            -0.038146      2020-01-31   
40   URBN 2020-01-31             0.096939      2021-01-31   
41   URBN 2021-01-31             0.085422      2022-01-31   
42   URBN 2022-01-31             0.143621      2023-01-31   
43   URBN 2023-01-31            -0.022923      2024-01-31   
44   URBN 2024-01-31            -0.138090      2025-01-31   
45   URBN 2025-01-31             0.051737      2026-01-31   

    next_gross_margin_change  next_ocf_margin_change  gross_margin_declined  \
38                  0.015802                0.029241                  False   
39                 -0.029668               -0.044300                   True   
40                 -0.061372                0.014099                   True   
41                  0.078584               -0.003858                  False   
42                 -0.030798               -0.049228   

In [77]:
paired.to_csv(
    PROCESSED_DIR / "pilot_next_year_pairing_audit.csv",
    index=False, date_format="%Y-%m-%d",
)
next_year_sample.to_csv(
    PROCESSED_DIR / "pilot_next_year_pairs.csv",
    index=False, date_format="%Y-%m-%d",
)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module4_metadata": module4_metadata,
    "research_design": "Retrospective association of annual inventory features with next fiscal-year outcomes.",
    "unit": "Ratios and changes stored as decimals; dates have daily precision.",
    "current_feature_rows": len(paired),
    "usable_pair_rows": len(next_year_sample),
    "signal_date_rule": "Selected annual filing date used as a proxy; not certified first public disclosure.",
    "pair_rule": "Consecutive complete target year, no known target scope break, current filing before target year end.",
    "target_overlap": "The target fiscal year may already have started when the current annual report is filed.",
    "gross_margin_target": "Next gross margin minus current gross margin.",
    "cash_flow_target": "Next OCF-to-revenue ratio minus current OCF-to-revenue ratio.",
    "accounting_scope_verified": False,
    "point_in_time_verified": False,
    "full_year_forecast_ready": False,
}
(PROCESSED_DIR / "pilot_module5_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_next_year_pairing_audit.csv")
print("Saved: pilot_next_year_pairs.csv")
print("Saved: pilot_module5_metadata.json")

Saved: pilot_next_year_pairing_audit.csv
Saved: pilot_next_year_pairs.csv
Saved: pilot_module5_metadata.json


## Module 6: Explore Inventory Signals and Review Pilot Cases

In [ ]:
%pip install matplotlib

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.size"] = 10
print("Project directory:", PROJECT_ROOT)

In [79]:
analysis = pd.read_csv(
    PROCESSED_DIR / "pilot_next_year_pairs.csv",
    parse_dates=["period_end", "next_period_end", "signal_available_date"],
    dtype={"gross_margin_declined": "boolean", "ocf_margin_declined": "boolean"},
)
module5_metadata = json.loads(
    (PROCESSED_DIR / "pilot_module5_metadata.json").read_text(encoding="utf-8")
)
analysis["inventory_gap_pp"] = analysis["inventory_sales_gap"] * 100
analysis["next_gross_margin_change_pp"] = analysis["next_gross_margin_change"] * 100
analysis["next_ocf_margin_change_pp"] = analysis["next_ocf_margin_change"] * 100
plot_columns = ["inventory_gap_pp", "next_gross_margin_change_pp", "next_ocf_margin_change_pp"]
source_caption = (
    f"Source: cached SEC Company Facts | {len(analysis)} annual pairs | "
    f"Current period ends: {analysis['period_end'].min().date()} to {analysis['period_end'].max().date()}"
)
print("Pilot paired rows:", len(analysis))
display(analysis[plot_columns].describe())

Pilot paired rows: 43


       inventory_gap_pp  next_gross_margin_change_pp  \
count         43.000000                    43.000000   
mean           1.471901                    -0.196574   
std           10.953235                     3.326170   
min          -33.553917                    -6.400384   
25%           -2.644227                    -2.635739   
50%            0.790332                     0.114075   
75%            6.055988                     1.448575   
max           32.910339                     9.207029   

       next_ocf_margin_change_pp  
count                  43.000000  
mean                   -0.272638  
std                     4.401804  
min                   -10.904632  
25%                    -2.929085  
50%                    -0.385798  
75%                     2.191384  
max                    15.327818  

In [80]:
company_summary = analysis.groupby("ticker").agg(
    paired_rows=("ticker", "size"),
    mean_inventory_gap_pp=("inventory_gap_pp", "mean"),
    mean_next_gross_margin_change_pp=("next_gross_margin_change_pp", "mean"),
    mean_next_ocf_margin_change_pp=("next_ocf_margin_change_pp", "mean"),
    gross_margin_decline_rate=("gross_margin_declined", "mean"),
    ocf_margin_decline_rate=("ocf_margin_declined", "mean"),
)
display(company_summary.round(3))

        paired_rows  mean_inventory_gap_pp  mean_next_gross_margin_change_pp  \
ticker                                                                         
AEO              13                  1.313                            -0.072   
ANF              15                  1.677                            -0.153   
URBN             15                  1.405                            -0.348   

        mean_next_ocf_margin_change_pp  gross_margin_decline_rate  \
ticker                                                              
AEO                             -0.363                      0.462   
ANF                              0.040                      0.533   
URBN                            -0.507                      0.467   

        ocf_margin_decline_rate  
ticker                           
AEO                       0.462  
ANF                       0.533  
URBN                        0.6  

In [81]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, outcome, title in zip(
    axes,
    ["next_gross_margin_change_pp", "next_ocf_margin_change_pp"],
    ["Next-year gross margin change", "Next-year OCF-to-revenue change"],
):
    for ticker, company in analysis.groupby("ticker"):
        ax.scatter(company["inventory_gap_pp"], company[outcome], label=ticker, alpha=0.8, s=45)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Inventory growth minus sales growth (pp)")
    ax.set_ylabel("Next-year ratio change (pp)")
    ax.set_title(title)
    ax.legend(title="Company")
fig.suptitle("Pilot annual associations")
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.06, 1, 0.93])
fig.savefig(FIGURE_DIR / "pilot_inventory_gap_scatter.png", dpi=180, bbox_inches="tight")
plt.show()

<Figure size 1300x500 with 2 Axes>

In [83]:
analysis["inventory_group"] = "Gap <= 0"
analysis.loc[analysis["inventory_sales_gap"] > 0, "inventory_group"] = "Gap > 0"
group_summary = analysis.groupby(["ticker", "inventory_group"]).agg(
    paired_rows=("ticker", "size"),
    mean_next_gross_margin_change_pp=("next_gross_margin_change_pp", "mean"),
    mean_next_ocf_margin_change_pp=("next_ocf_margin_change_pp", "mean"),
    gross_margin_decline_rate=("gross_margin_declined", "mean"),
    ocf_margin_decline_rate=("ocf_margin_declined", "mean"),
)
display(group_summary.round(3))
group_order = ["Gap <= 0", "Gap > 0"]
group_counts = group_summary["paired_rows"].unstack("inventory_group").reindex(columns=group_order)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, title in zip(
    axes,
    ["mean_next_gross_margin_change_pp", "mean_next_ocf_margin_change_pp"],
    ["Mean next-year gross margin change", "Mean next-year OCF-to-revenue change"],
):
    means = group_summary[metric].unstack("inventory_group").reindex(columns=group_order)
    means.plot.bar(ax=ax, rot=0, color=["gray", "tab:blue"])
    for group_name, bars in zip(group_order, ax.containers):
        labels = [f"n={int(n)}" if pd.notna(n) else "" for n in group_counts[group_name]]
        ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Company")
    ax.set_ylabel("Mean next-year ratio change (pp)")
    ax.margins(y=0.2)
    ax.legend(title="Current inventory gap")
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIGURE_DIR / "pilot_inventory_group_means.png", dpi=180, bbox_inches="tight")
plt.show()

                        paired_rows  mean_next_gross_margin_change_pp  \
ticker inventory_group                                                  
AEO    Gap <= 0                   5                            -1.588   
       Gap > 0                    8                             0.875   
ANF    Gap <= 0                   6                             1.118   
       Gap > 0                    9                            -1.001   
URBN   Gap <= 0                   7                            -0.278   
       Gap > 0                    8                            -0.408   

                        mean_next_ocf_margin_change_pp  \
ticker inventory_group                                   
AEO    Gap <= 0                                 -1.644   
       Gap > 0                                   0.437   
ANF    Gap <= 0                                  1.877   
       Gap > 0                                  -1.184   
URBN   Gap <= 0                                 -0.918   
       Ga

<Figure size 1300x500 with 2 Axes>

In [84]:
sensitivity_rows = []
for sample_name, sample in [
    ("All pairs", analysis),
    ("Unchanged period lengths", analysis.loc[~analysis["pair_period_length_changed"]]),
]:
    for ticker, company in sample.groupby("ticker"):
        for outcome in ["next_gross_margin_change_pp", "next_ocf_margin_change_pp"]:
            correlation = float("nan")
            if company["inventory_gap_pp"].nunique() > 1 and company[outcome].nunique() > 1:
                correlation = company["inventory_gap_pp"].corr(company[outcome])
            sensitivity_rows.append({
                "sample": sample_name, "ticker": ticker, "outcome": outcome,
                "paired_rows": len(company), "pearson_r": correlation,
            })
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))

                      sample ticker                      outcome  paired_rows  \
0                  All pairs    AEO  next_gross_margin_change_pp           13   
1                  All pairs    AEO    next_ocf_margin_change_pp           13   
2                  All pairs    ANF  next_gross_margin_change_pp           15   
3                  All pairs    ANF    next_ocf_margin_change_pp           15   
4                  All pairs   URBN  next_gross_margin_change_pp           15   
5                  All pairs   URBN    next_ocf_margin_change_pp           15   
6   Unchanged period lengths    AEO  next_gross_margin_change_pp            6   
7   Unchanged period lengths    AEO    next_ocf_margin_change_pp            6   
8   Unchanged period lengths    ANF  next_gross_margin_change_pp            6   
9   Unchanged period lengths    ANF    next_ocf_margin_change_pp            6   
10  Unchanged period lengths   URBN  next_gross_margin_change_pp           15   
11  Unchanged period lengths

In [86]:
cases = analysis.sort_values("inventory_sales_gap", ascending=False).drop_duplicates("ticker").copy()
cases["cik"] = cases["ticker"].map({"AEO": 919012, "ANF": 1018840, "URBN": 912615})
for index, row in cases.iterrows():
    cik = int(row["cik"])
    cases.loc[index, "current_filing_url"] = (
        f"https://www.sec.gov/Archives/edgar/data/{cik}/{row['accn'].replace('-', '')}/"
    )
    cases.loc[index, "next_filing_url"] = (
        f"https://www.sec.gov/Archives/edgar/data/{cik}/{row['next_accn'].replace('-', '')}/"
    )
cases["case_review_status"] = "pending"
cases["review_notes"] = ""
case_columns = [
    "ticker", "period_end", "next_period_end", "inventory_gap_pp",
    "next_gross_margin_change_pp", "next_ocf_margin_change_pp",
    "current_filing_url", "next_filing_url", "case_review_status", "review_notes",
]
cases = cases[case_columns]
display(cases)

   ticker period_end next_period_end  inventory_gap_pp  \
16    ANF 2014-02-01      2015-01-31         32.910339   
4     AEO 2017-01-28      2018-02-03         14.955560   
39   URBN 2022-01-31      2023-01-31         14.362091   

    next_gross_margin_change_pp  next_ocf_margin_change_pp  \
16                    -0.764099                   4.083338   
4                     -1.758215                   0.264112   
39                    -3.079838                  -4.922798   

                                   current_filing_url  \
16  https://www.sec.gov/Archives/edgar/data/101884...   
4   https://www.sec.gov/Archives/edgar/data/919012...   
39  https://www.sec.gov/Archives/edgar/data/912615...   

                                      next_filing_url case_review_status  \
16  https://www.sec.gov/Archives/edgar/data/101884...            pending   
4   https://www.sec.gov/Archives/edgar/data/919012...            pending   
39  https://www.sec.gov/Archives/edgar/data/912615...        

In [88]:
outputs = {
    "pilot_company_eda_summary.csv": company_summary.reset_index(),
    "pilot_inventory_group_summary.csv": group_summary.reset_index(),
    "pilot_period_length_sensitivity.csv": sensitivity,
    "pilot_case_review.csv": cases,
}
for filename, data in outputs.items():
    data.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
    print("Saved:", filename)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module5_metadata": module5_metadata,
    "paired_rows": len(analysis),
    "company_count": int(analysis["ticker"].nunique()),
    "group_rule": "Within each company: inventory-sales growth gap > 0 versus <= 0.",
    "sensitivity_rule": "All pairs versus pairs without flagged annual-period length changes.",
    "case_rule": "Highest current inventory-sales growth gap per company; filing review pending.",
    "display_units": "Changes in percentage points; decline rates stored as fractions.",
    "statistical_inference_performed": False,
    "accounting_scope_verified": False,
    "full_year_forecast_ready": False,
}
(PROCESSED_DIR / "pilot_module6_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_module6_metadata.json")
print("Figure directory:", FIGURE_DIR)

Saved: pilot_company_eda_summary.csv
Saved: pilot_inventory_group_summary.csv
Saved: pilot_period_length_sensitivity.csv
Saved: pilot_case_review.csv
Saved: pilot_module6_metadata.json
Figure directory: [project root]/outputs/figures


## Module 7: Expand the Candidate Sample and Screen Financial Coverage

In [90]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import requests
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [91]:
universe= pd.read_csv(
    PROCESSED_DIR / "retail_candidate_universe.csv", dtype={"cik10": "string"}
)
candidate_pool = universe.loc[
    universe["identity_status"].eq("matched") & universe["cik10"].notna()
].drop_duplicates("ticker").copy()
MIN_CORE_PERIODS = 5
print("Identity-matched candidates:", len(candidate_pool))
display(candidate_pool.groupby("subgroup", sort=False).size().rename("Candidate Count"))

Identity-matched candidates: 42


subgroup
Apparel, Footwear, and Off-Price Retail               23
Department Stores                                      3
Home Furnishings, Furniture, and Home Improvement      7
Sporting Goods, Electronics, and Other Merchandise     6
Beauty and Personal Care                               3
Name: Candidate Count, dtype: int64

In [92]:
all_company_facts = {}
snapshot_times = {}
download_rows = []
download_allowed = True
for index, row in candidate_pool.iterrows():
    ticker = row["ticker"]
    cache_path = FACTS_DIR / f"CIK{row['cik10']}_companyfacts.json"
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{row['cik10']}.json"
    status, error, snapshot = "not_attempted", "", None
    try:
        if cache_path.exists():
            snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
            status = "cached"
        elif download_allowed:
            try:
                response = requests.get(url, headers=SEC_HEADERS, timeout=30)
            finally:
                time.sleep(1)
            if response.status_code in [403, 429]:
                download_allowed = False
            response.raise_for_status()
            snapshot = {
                "source_url": url,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                "payload": response.json(),
            }
            status = "downloaded"
        if snapshot is not None:
            payload = snapshot["payload"]
            if int(payload["cik"]) != int(row["cik10"]) or not isinstance(payload["facts"], dict):
                raise ValueError("Invalid Company Facts payload or CIK mismatch.")
            if status == "downloaded":
                cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
            snapshot_times[ticker] = snapshot["retrieved_at_utc"]
            all_company_facts[ticker] = payload
    except (requests.RequestException, ValueError, KeyError, OSError) as exc:
        status, error = "failed", str(exc)
    download_rows.append({"ticker": ticker, "download_status": status, "error": error})
    print(ticker, status)
download_log = pd.DataFrame(download_rows)
display(download_log["download_status"].value_counts())
display(download_log.loc[download_log["download_status"].isin(["failed", "not_attempted"])])

TJX downloaded
ROST downloaded
BURL downloaded
LULU downloaded
GAP downloaded
URBN cached
ANF cached
BOOT downloaded
AEO cached
BKE downloaded
CTRN downloaded
JILL downloaded
SFIX downloaded
SHOE downloaded
LE downloaded
CURV downloaded
ZUMZ downloaded
DLTH downloaded
TLYS downloaded
AKA downloaded
CATO downloaded
LVLU downloaded
DXLG downloaded
M downloaded
KSS downloaded
DDS downloaded
HD downloaded
LOW downloaded
FND downloaded
HVT downloaded
WSM downloaded
RH downloaded
ARHS downloaded
DKS downloaded
ASO downloaded
SPWH downloaded
BBY downloaded
GME downloaded
FIVE downloaded
ULTA downloaded
SBH downloaded
BBWI downloaded


download_status
downloaded    39
cached         3
Name: count, dtype: int64

Empty DataFrame
Columns: [ticker, download_status, error]
Index: []

In [98]:
pilot_metadata = json.loads(
    (PROCESSED_DIR / "pilot_annual_panel_metadata.json").read_text(encoding="utf-8")
)
tag_candidates = pilot_metadata["tag_candidates_in_priority_order"]
metrics = list(tag_candidates)
company_tag_additions = {
    "revenue": {
        "TJX": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
        "RH": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
        "TLYS": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
    },
    "inventory": {
        "GAP": ["InventoryFinishedGoodsNetOfReserves"],
        "DDS": ["RetailRelatedInventoryMerchandise"],
        "RH": ["RetailRelatedInventoryMerchandise"],
        "SPWH": ["RetailRelatedInventory"],
        "ULTA": ["InventoryFinishedGoods"],
    },
}
print("Cost tag candidates:", tag_candidates["cost_of_sales"])
rows = []
for ticker, data in all_company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, base_tags in tag_candidates.items():
        possible_tags = base_tags + company_tag_additions.get(metric, {}).get(ticker, [])
        for priority, tag in enumerate(possible_tags):
            for fact in tags.get(tag, {}).get("units", {}).get("USD", []):
                rows.append({
                    "ticker": ticker, "metric": metric, "tag": tag, "priority": priority,
                    "start": fact.get("start"), "end": fact["end"], "value": fact["val"],
                    "form": fact["form"], "filed": fact["filed"], "accn": fact["accn"],
                })
candidate_facts = pd.DataFrame(rows).drop_duplicates()
if candidate_facts.empty:
    raise ValueError("No candidate USD facts found. Review the download log and tags.")
for column in ["start", "end", "filed"]:
    candidate_facts[column] = pd.to_datetime(candidate_facts[column])
display(candidate_facts.head())

Cost tag candidates: ['CostOfRevenue', 'CostOfGoodsAndServicesSold', 'CostOfGoodsSold', 'CostOfGoodsSoldExcludingDepreciationDepletionAndAmortization', 'CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization']


  ticker   metric              tag  priority      start        end  \
0    TJX  revenue  SalesRevenueNet         1 2013-02-03 2014-02-01   
1    TJX  revenue  SalesRevenueNet         1 2014-02-02 2014-05-03   
2    TJX  revenue  SalesRevenueNet         1 2014-05-04 2014-08-02   
3    TJX  revenue  SalesRevenueNet         1 2014-02-02 2014-11-01   
4    TJX  revenue  SalesRevenueNet         1 2014-08-03 2014-11-01   

         value  form      filed                  accn  
0  27422696000  10-K 2016-03-29  0001193125-16-521424  
1   6491176000  10-K 2016-03-29  0001193125-16-521424  
2   6917212000  10-K 2016-03-29  0001193125-16-521424  
3  20774454000  10-Q 2015-12-01  0001193125-15-391899  
4   7366066000  10-K 2016-03-29  0001193125-16-521424  

In [99]:
candidate_facts["period_days"] = (
    candidate_facts["end"] - candidate_facts["start"]
).dt.days + 1
annual_revenue = candidate_facts.loc[
    candidate_facts["metric"].eq("revenue")
    & candidate_facts["form"].isin(["10-K", "10-K/A"])
    & candidate_facts["period_days"].between(350, 380)
]
candidate_anchors = annual_revenue.sort_values(["ticker", "end", "filed", "accn", "priority"])
candidate_anchors = candidate_anchors.drop_duplicates(["ticker", "end"], keep="first")
if candidate_anchors.empty:
    raise ValueError("No annual revenue anchors found. Review the candidate revenue tags.")
print("Companies with annual revenue anchors:", candidate_anchors["ticker"].nunique())
print("Candidate annual periods:", len(candidate_anchors))

Companies with annual revenue anchors: 42
Candidate annual periods: 656


In [100]:
annual_rows, source_rows = [], []
for index, year in candidate_anchors.iterrows():
    annual_row = {
        "ticker": year["ticker"], "period_start": year["start"], "period_end": year["end"],
        "period_days": year["period_days"], "filed": year["filed"], "accn": year["accn"],
    }
    for metric in metrics:
        matches = candidate_facts.loc[
            candidate_facts["ticker"].eq(year["ticker"]) & candidate_facts["metric"].eq(metric)
            & candidate_facts["end"].eq(year["end"]) & candidate_facts["accn"].eq(year["accn"])
        ]
        if metric in ["inventory", "total_assets"]:
            matches = matches.loc[matches["start"].isna()]
        else:
            matches = matches.loc[matches["start"].eq(year["start"])]
        value, tag, status = None, None, "missing"
        if not matches.empty:
            tag = matches.sort_values("priority").iloc[0]["tag"]
            chosen = matches.loc[matches["tag"].eq(tag)]
            if chosen["value"].nunique() == 1:
                value, status = chosen.iloc[0]["value"], "selected_pending_review"
            else:
                status = "ambiguous"
        annual_row[metric] = value
        source_rows.append({
            "ticker": year["ticker"], "period_end": year["end"], "metric": metric,
            "tag": tag, "value": value, "status": status, "accn": year["accn"],
        })
    annual_rows.append(annual_row)
candidate_panel = pd.DataFrame(annual_rows).sort_values(["ticker", "period_end"])
candidate_sources = pd.DataFrame(source_rows)
display(candidate_panel.head())

  ticker period_start period_end  period_days      filed  \
0    AEO   2008-02-03 2009-01-31        364.0 2011-03-11   
1    AEO   2009-02-01 2010-01-30        364.0 2011-03-11   
2    AEO   2010-01-31 2011-01-29        364.0 2011-03-11   
3    AEO   2011-01-30 2012-01-28        364.0 2012-03-15   
4    AEO   2012-01-29 2013-02-02        371.0 2013-03-12   

                   accn     revenue  cost_of_sales  gross_profit    inventory  \
0  0000950123-11-024788  2948679000            NaN  1.197186e+09          NaN   
1  0000950123-11-024788  2940269000            NaN  1.173430e+09  326454000.0   
2  0000950123-11-024788  2967559000            NaN  1.170959e+09  301208000.0   
3  0001193125-12-117468  3159818000            NaN  1.128341e+09  378426000.0   
4  0001193125-13-102293  3475802000            NaN  1.390322e+09  332452000.0   

   total_assets  operating_cash_flow  
0           NaN          345112000.0  
1  2.138148e+09          400326000.0  
2  1.879998e+09          402594000.

In [102]:
coverage_periods = candidate_panel[["ticker", "period_end"]].copy()
for metric in metrics:
    coverage_periods[metric] = candidate_panel[metric].notna()
coverage_periods["core_complete"] = candidate_panel[
    ["revenue", "inventory", "total_assets", "operating_cash_flow"]
].notna().all(axis=1)
coverage_periods["six_metric_complete"] = candidate_panel[metrics].notna().all(axis=1)
company_coverage = coverage_periods.groupby("ticker").agg(
    annual_periods=("period_end", "size"), first_period_end=("period_end", "min"),
    last_period_end=("period_end", "max"), core_complete_periods=("core_complete", "sum"),
    six_metric_complete_periods=("six_metric_complete", "sum"),
)
for metric in metrics:
    company_coverage[f"{metric}_periods"] = coverage_periods.groupby("ticker")[metric].sum()
coverage = candidate_pool[["ticker", "subgroup", "cik10", "sec_company_name"]].merge(
    download_log, on="ticker", how="left"
).merge(company_coverage, on="ticker", how="left")
coverage["coverage_status"] = "download_not_ready"
loaded = coverage["download_status"].isin(["cached", "downloaded"])
coverage.loc[loaded, "coverage_status"] = "no_annual_revenue_anchor"
coverage.loc[coverage["annual_periods"].notna(), "coverage_status"] = "needs_metric_review"
coverage.loc[coverage["core_complete_periods"].ge(MIN_CORE_PERIODS), "coverage_status"] = "ready_for_accounting_review"
review_sheet = coverage.copy()
review_sheet["accounting_review_status"] = "pending"
review_sheet["include_decision"] = "pending"
review_sheet["business_scope_notes"] = ""
review_sheet["cost_and_gross_profit_notes"] = ""
print("Final company eligibility remains pending.")
display(coverage["coverage_status"].value_counts())
display(coverage[["ticker", "annual_periods", "core_complete_periods", "six_metric_complete_periods", "coverage_status"]])

Final company eligibility remains pending.


coverage_status
ready_for_accounting_review    42
Name: count, dtype: int64

   ticker  annual_periods  core_complete_periods  six_metric_complete_periods  \
0     TJX              19                     18                            0   
1    ROST              18                     17                            0   
2    BURL              15                     14                            0   
3    LULU              18                     13                           13   
4     GAP              19                     14                            7   
5    URBN              18                     17                           17   
6     ANF              18                     17                           15   
7    BOOT              14                     13                           13   
8     AEO              18                     17                           11   
9     BKE              18                     17                            7   
10   CTRN              17                     16                            3   
11   JILL              11   

In [104]:
output_tables = {
    "candidate_download_log.csv": download_log,
    "candidate_annual_panel_preliminary.csv": candidate_panel,
    "candidate_field_sources.csv": candidate_sources,
    "candidate_financial_coverage.csv": coverage,
}
for filename, table in output_tables.items():
    table.to_csv(PROCESSED_DIR / filename, index=False)
review_path = PROCESSED_DIR / "candidate_scope_review.csv"
review_columns = [
    "accounting_review_status", "include_decision",
    "business_scope_notes", "cost_and_gross_profit_notes",
]
if review_path.exists():
    previous_review = pd.read_csv(
        review_path, dtype={column: "string" for column in review_columns}
    )
    review_sheet = coverage.merge(
        previous_review[["ticker"] + review_columns], on="ticker", how="left", validate="one_to_one"
    )
    for column in review_columns:
        default = "pending" if column in ["accounting_review_status", "include_decision"] else ""
        review_sheet[column] = review_sheet[column].fillna(default)
review_sheet.to_csv(review_path, index=False)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_rows": len(candidate_pool), "loaded_companies": len(all_company_facts),
    "companies_with_annual_anchors": candidate_panel["ticker"].nunique(),
    "annual_periods": len(candidate_panel), "snapshot_times": snapshot_times,
    "tag_candidates_in_priority_order": tag_candidates, "annual_period_days": [350, 380],
    "company_tag_additions": company_tag_additions, "extraction_version": "module7_rev2",
    "min_core_periods_for_review": MIN_CORE_PERIODS, "unit": "USD",
    "filing_rule": "Earliest visible annual revenue filing in the cached data; same-accession metric matching",
    "derivations_applied": False, "accounting_scope_verified": False,
    "final_sample_selected": False, "point_in_time_certified": False,
}
(PROCESSED_DIR / "candidate_module7_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Screening outputs saved in:", PROCESSED_DIR)

Screening outputs saved in: [project root]/work/data/processed


## Module 8: Review Accounting Scope and Build Outcome-Specific Working Samples

In [105]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
print("Project directory:", PROJECT_ROOT)

Project directory: [project root]


In [107]:
audit = pd.read_csv(
    PROCESSED_DIR / "candidate_annual_panel_preliminary.csv",
    parse_dates=["period_start", "period_end", "filed"],
).sort_values(["ticker", "period_end"]).reset_index(drop=True)
sources = pd.read_csv(
    PROCESSED_DIR / "candidate_field_sources.csv", parse_dates=["period_end"]
)
sources["method"] = ""
sources.loc[sources["value"].notna(), "method"] = "sec_tag"
company_review = pd.read_csv(
    PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False
)
module7_metadata = json.loads(
    (PROCESSED_DIR / "candidate_module7_metadata.json").read_text(encoding="utf-8")
)
cik_lookup = company_review.set_index("ticker")["cik10"]
metrics = ["revenue", "cost_of_sales", "gross_profit", "inventory", "total_assets", "operating_cash_flow"]
MIN_ANNUAL_ROWS = 5
print("Expanded annual rows:", len(audit))
display(audit[metrics].isna().sum().rename("Missing Rows").to_frame())

Expanded annual rows: 656


                     Missing Rows
revenue                         0
cost_of_sales                  74
gross_profit                  110
inventory                      63
total_assets                   42
operating_cash_flow             2

In [108]:
additional_cost_tag = "CostOfMerchandiseSalesBuyingAndOccupancyCosts"
for ticker in ["GAP", "BKE", "DXLG"]:
    path = FACTS_DIR / f"CIK{cik_lookup.loc[ticker]}_companyfacts.json"
    payload = json.loads(path.read_text(encoding="utf-8"))["payload"]
    records = payload["facts"]["us-gaap"].get(additional_cost_tag, {}).get("units", {}).get("USD", [])
    missing = audit.loc[audit["ticker"].eq(ticker) & audit["cost_of_sales"].isna()]
    for index, row in missing.iterrows():
        values = [fact["val"] for fact in records
                  if fact.get("start") == row["period_start"].strftime("%Y-%m-%d")
                  and fact["end"] == row["period_end"].strftime("%Y-%m-%d")
                  and fact["accn"] == row["accn"]]
        values = list(set(values))
        if len(values) != 1 or pd.isna(row["gross_profit"]):
            continue
        value = values[0]
        if value < 0 or abs(row["revenue"] - value - row["gross_profit"]) > 1:
            continue
        match = (sources["ticker"].eq(ticker) & sources["period_end"].eq(row["period_end"])
                 & sources["accn"].eq(row["accn"]) & sources["metric"].eq("cost_of_sales"))
        if not sources.loc[match, "status"].eq("missing").all():
            continue
        audit.at[index, "cost_of_sales"] = value
        sources.loc[match, ["tag", "value", "status", "method"]] = [additional_cost_tag, value, "selected_additional_tag_pending_review", "sec_tag"]
print("Additional cost values selected:", int(sources["status"].eq("selected_additional_tag_pending_review").sum()))

Additional cost values selected: 25


In [110]:
pilot_sources = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_field_sources.csv", parse_dates=["period_end"]
)
for index, old in pilot_sources.loc[pilot_sources["status"].eq("derived_pending_review")].iterrows():
    match = (audit["ticker"].eq(old["ticker"]) & audit["period_end"].eq(old["period_end"])
             & audit["accn"].eq(old["accn"]))
    if not match.any():
        continue
    row_index = audit.index[match][0]
    row = audit.loc[row_index]
    metric = old["metric"]
    if pd.notna(row[metric]):
        continue
    if old["ticker"] == "AEO" and metric == "cost_of_sales":
        value = row["revenue"] - row["gross_profit"]
    elif old["ticker"] == "ANF" and metric == "gross_profit":
        value = row["revenue"] - row["cost_of_sales"]
    else:
        continue
    if pd.isna(value) or value != old["value"]:
        continue
    source_match = (sources["ticker"].eq(old["ticker"]) & sources["period_end"].eq(old["period_end"])
                    & sources["accn"].eq(old["accn"]) & sources["metric"].eq(metric))
    if not sources.loc[source_match, "status"].eq("missing").all():
        continue
    audit.at[row_index, metric] = value
    sources.loc[source_match, ["value", "status", "method"]] = [value, "derived_pilot_rule_pending_review", old["derivation"]]
print("Pilot derived values restored:", int(sources["status"].eq("derived_pilot_rule_pending_review").sum()))
display(audit[metrics].isna().sum().rename("Remaining Missing Rows").to_frame())

Pilot derived values restored: 9


                     Remaining Missing Rows
revenue                                   0
cost_of_sales                            42
gross_profit                            108
inventory                                63
total_assets                             42
operating_cash_flow                       2

In [111]:
scope_breaks = pd.DataFrame([
    {"ticker": "AEO", "period_end": "2013-02-02", "scope_reason": "Discontinued operations and comparative reclassification",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm"},
    {"ticker": "BBWI", "period_end": "2022-01-29", "scope_reason": "Victoria's Secret separation and discontinued operations",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/701985/000070198522000009/bbwi-20220129.htm"},
    {"ticker": "DKS", "period_end": "2026-01-31", "scope_reason": "Foot Locker acquisition changes the consolidation scope",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/1089063/000108906326000007/dks-20260131.htm"},
])
scope_breaks["period_end"] = pd.to_datetime(scope_breaks["period_end"])
audit = audit.merge(scope_breaks, on=["ticker", "period_end"], how="left", validate="one_to_one")
audit["known_scope_break"] = audit["scope_reason"].notna()
derived = sources.loc[sources["status"].eq("derived_pilot_rule_pending_review")]
derived_counts = derived.groupby(["ticker", "period_end"]).size().rename("derived_fields")
audit = audit.merge(derived_counts, on=["ticker", "period_end"], how="left")
audit["derived_fields"] = audit["derived_fields"].fillna(0).astype(int)
audit["gross_profit_bridge_gap"] = audit["revenue"] - audit["cost_of_sales"] - audit["gross_profit"]
audit["bridge_needs_review"] = audit["gross_profit_bridge_gap"].abs() > 1
audit["base_values_ok"] = (
    audit["revenue"].gt(0) & audit["total_assets"].gt(0)
    & audit["inventory"].ge(0) & audit["inventory"].le(audit["total_assets"])
)
audit["gm_row_usable"] = audit["base_values_ok"] & audit["gross_profit"].notna()
audit["ocf_row_usable"] = audit["base_values_ok"] & audit["operating_cash_flow"].notna()
audit["gross_margin"] = audit["gross_profit"] / audit["revenue"]
audit["ocf_margin"] = audit["operating_cash_flow"] / audit["revenue"]
audit.loc[~audit["gm_row_usable"], "gross_margin"] = float("nan")
audit.loc[~audit["ocf_row_usable"], "ocf_margin"] = float("nan")
display(audit.loc[audit["known_scope_break"], ["ticker", "period_end", "scope_reason"]])
display(audit.loc[audit["bridge_needs_review"], ["ticker", "period_end", "gross_profit_bridge_gap", "derived_fields"]])

    ticker period_end                                       scope_reason
4      AEO 2013-02-02  Discontinued operations and comparative reclas...
72    BBWI 2022-01-29  Victoria's Secret separation and discontinued ...
218    DKS 2026-01-31  Foot Locker acquisition changes the consolidat...

    ticker period_end  gross_profit_bridge_gap  derived_fields
80     BBY 2011-02-26               24000000.0               0
81     BBY 2012-03-03               19000000.0               0
84     BBY 2016-01-30                3000000.0               0
194    DDS 2020-02-01              139691000.0               0
195    DDS 2021-01-30              132290000.0               0
196    DDS 2022-01-29              131274000.0               0
197    DDS 2023-01-28              125134000.0               0
614   URBN 2020-01-31               14611000.0               0
615   URBN 2021-01-31               15496000.0               0
617   URBN 2023-01-31                6417000.0               0
618   URBN 2024-01-31               11875000.0               0
619   URBN 2025-01-31                4601000.0               0
620   URBN 2026-01-31                1989000.0               0

In [112]:
queue = sources.loc[
    sources["metric"].isin(["cost_of_sales", "gross_profit"])
    & sources["status"].isin(["missing", "ambiguous"]),
    ["ticker", "period_end", "accn", "metric", "status"],
].merge(audit[["ticker", "period_end", "accn", "period_start"] + metrics],
        on=["ticker", "period_end", "accn"], how="left", validate="many_to_one")
queue["proposed_value"] = float("nan")
cost_missing = queue["metric"].eq("cost_of_sales")
queue.loc[cost_missing, "proposed_value"] = queue["revenue"] - queue["gross_profit"]
queue.loc[~cost_missing, "proposed_value"] = queue["revenue"] - queue["cost_of_sales"]
queue.loc[cost_missing & queue["proposed_value"].lt(0), "proposed_value"] = float("nan")
queue["current_filing_url"] = (
    "https://www.sec.gov/Archives/edgar/data/"
    + queue["ticker"].map(cik_lookup).astype(int).astype(str) + "/"
    + queue["accn"].str.replace("-", "", regex=False) + "/"
)
queue["review_status"] = "pending"
queue["reviewed_value"] = float("nan")
queue["evidence_url"] = ""
queue["review_notes"] = ""
queue = queue.sort_values(["period_end", "ticker"], ascending=[False, True])
print("Cost and gross profit fields awaiting review:", len(queue))
display(queue.groupby(["ticker", "metric"]).size().rename("Review Fields").to_frame())
display(queue[["ticker", "period_end", "metric", "proposed_value", "current_filing_url"]].head(8))

Cost and gross profit fields awaiting review: 150


                      Review Fields
ticker metric                      
BKE    cost_of_sales              4
BURL   gross_profit              15
CATO   gross_profit              17
CTRN   gross_profit              13
DDS    gross_profit               3
FIVE   gross_profit               3
GAP    cost_of_sales              1
HVT    cost_of_sales              8
JILL   cost_of_sales              1
       gross_profit               1
KSS    gross_profit               8
LE     gross_profit               3
M      gross_profit               8
ROST   gross_profit              18
TJX    cost_of_sales             11
       gross_profit              19
WSM    cost_of_sales             17

    ticker period_end        metric  proposed_value  \
18    BURL 2026-01-31  gross_profit    5.062685e+09   
35    CATO 2026-01-31  gross_profit    2.222610e+08   
48    CTRN 2026-01-31  gross_profit    3.246420e+08   
54    FIVE 2026-01-31  gross_profit    1.714686e+09   
73     KSS 2026-01-31  gross_profit    6.299000e+09   
84       M 2026-01-31  gross_profit    8.267000e+09   
102   ROST 2026-01-31  gross_profit    6.303303e+09   
132    TJX 2026-01-31  gross_profit    1.869300e+10   

                                    current_filing_url  
18   https://www.sec.gov/Archives/edgar/data/157929...  
35   https://www.sec.gov/Archives/edgar/data/18255/...  
48   https://www.sec.gov/Archives/edgar/data/131848...  
54   https://www.sec.gov/Archives/edgar/data/117760...  
73   https://www.sec.gov/Archives/edgar/data/885639...  
84   https://www.sec.gov/Archives/edgar/data/794367...  
102  https://www.sec.gov/Archives/edgar/data/745732...  
132  https://www.sec.gov/Archives/edgar/data/109

In [113]:
counts = audit.groupby("ticker").agg(
    annual_rows=("period_end", "size"), gm_usable_years=("gm_row_usable", "sum"),
    ocf_usable_years=("ocf_row_usable", "sum"), known_scope_breaks=("known_scope_break", "sum"),
    bridge_review_rows=("bridge_needs_review", "sum"), derived_fields=("derived_fields", "sum"),
)
review_columns = [
    "ticker", "subgroup", "cik10", "sec_company_name", "accounting_review_status",
    "include_decision", "business_scope_notes", "cost_and_gross_profit_notes",
]
readiness = company_review[review_columns].merge(counts, on="ticker", how="left", validate="one_to_one")
not_excluded = readiness["include_decision"].ne("exclude")
readiness["gm_working_candidate"] = readiness["gm_usable_years"].ge(MIN_ANNUAL_ROWS) & not_excluded
readiness["ocf_working_candidate"] = readiness["ocf_usable_years"].ge(MIN_ANNUAL_ROWS) & not_excluded
gm_companies = readiness.loc[readiness["gm_working_candidate"], "ticker"]
ocf_companies = readiness.loc[readiness["ocf_working_candidate"], "ticker"]
gm_working_panel = audit.loc[audit["gm_row_usable"] & audit["ticker"].isin(gm_companies)].copy()
ocf_working_panel = audit.loc[audit["ocf_row_usable"] & audit["ticker"].isin(ocf_companies)].copy()
print("Gross margin working companies:", len(gm_companies), "Annual rows:", len(gm_working_panel))
print("Operating cash flow working companies:", len(ocf_companies), "Annual rows:", len(ocf_working_panel))
display(readiness[["ticker", "gm_usable_years", "ocf_usable_years", "gm_working_candidate", "ocf_working_candidate", "include_decision"]])

Gross margin working companies: 36 Annual rows: 488
Operating cash flow working companies: 42 Annual rows: 592


   ticker  gm_usable_years  ocf_usable_years  gm_working_candidate  \
0     TJX                0                18                 False   
1    ROST                0                17                 False   
2    BURL                0                14                 False   
3    LULU               13                13                  True   
4     GAP               14                14                  True   
5    URBN               17                17                  True   
6     ANF               17                17                  True   
7    BOOT               13                13                  True   
8     AEO               17                17                  True   
9     BKE               17                17                  True   
10   CTRN                3                16                 False   
11   JILL               10                10                  True   
12   SFIX               10                10                  True   
13   SHOE           

In [114]:
outputs = {
    "expanded_annual_audit.csv": audit,
    "expanded_field_sources.csv": sources,
    "expanded_missing_metric_queue.csv": queue,
    "expanded_company_readiness.csv": readiness,
    "expanded_gross_margin_working_panel.csv": gm_working_panel,
    "expanded_ocf_working_panel.csv": ocf_working_panel,
    "expanded_known_scope_breaks.csv": scope_breaks,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
decisions_path = PROCESSED_DIR / "expanded_metric_review_decisions.csv"
if not decisions_path.exists():
    queue.to_csv(decisions_path, index=False, date_format="%Y-%m-%d")
else:
    print("Existing manual metric decisions retained:", decisions_path.name)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module7_metadata": module7_metadata,
    "annual_rows": len(audit), "unit": "USD; ratios stored as decimals",
    "additional_cost_tag": additional_cost_tag, "additional_cost_companies": ["GAP", "BKE", "DXLG"],
    "restored_pilot_derivations": len(derived), "min_annual_rows": MIN_ANNUAL_ROWS,
    "gm_working_companies": len(gm_companies), "gm_working_annual_rows": len(gm_working_panel),
    "ocf_working_companies": len(ocf_companies), "ocf_working_annual_rows": len(ocf_working_panel),
    "candidate_eligibility_count": int((readiness["accounting_review_status"].eq("reviewed") & readiness["include_decision"].eq("include")).sum()),
    "known_scope_break_rule": "Flag AEO 2013-02-02, BBWI 2022-01-29 and DKS 2026-01-31; other changes remain subject to review",
    "formula_proposals_applied": False, "manual_metric_decisions_applied": False,
    "accounting_scope_verified": False, "point_in_time_verified": False, "next_year_pairs_built": False,
}
(PROCESSED_DIR / "expanded_module8_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Module 8 outputs saved in:", PROCESSED_DIR)

Module 8 outputs saved in: [project root]/work/data/processed


## Module 9: Reconcile Accounting Differences and Build Expanded Next-Year Pairs

In [115]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)

Project directory: [project root]
Processed data directory: [project root]/work/data/processed


In [116]:
annual = pd.read_csv(
    PROCESSED_DIR / "expanded_annual_audit.csv",
    parse_dates=["period_start", "period_end", "filed"],
).sort_values(["ticker", "period_end"]).reset_index(drop=True)
assert not annual.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year rows."
readiness = pd.read_csv(PROCESSED_DIR / "expanded_company_readiness.csv")
company_review = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", keep_default_na=False)
annual = annual.merge(
    company_review[["ticker", "subgroup", "accounting_review_status", "include_decision"]],
    on="ticker", how="left", validate="many_to_one",
)
module8_metadata = json.loads(
    (PROCESSED_DIR / "expanded_module8_metadata.json").read_text(encoding="utf-8")
)
print("Annual rows:", len(annual))
print("Candidate companies:", annual["ticker"].nunique())
display(annual[["ticker", "period_end", "revenue", "gross_profit", "operating_cash_flow"]].head())

Annual rows: 656
Candidate companies: 42


  ticker period_end     revenue  gross_profit  operating_cash_flow
0    AEO 2009-01-31  2948679000  1.197186e+09          345112000.0
1    AEO 2010-01-30  2940269000  1.173430e+09          400326000.0
2    AEO 2011-01-29  2967559000  1.170959e+09          402594000.0
3    AEO 2012-01-28  3159818000  1.128341e+09          239256000.0
4    AEO 2013-02-02  3475802000  1.390322e+09          499671000.0

In [117]:
sec_archive = "https://www.sec.gov/Archives/edgar/data/"
evidence = pd.DataFrame([
    ["BBY", "2011-02-26", "0001047469-11-004045", None, 24_000_000, "764478/000104746911004045/a2203505z10-k.htm", "Restructuring charges in cost of goods sold"],
    ["BBY", "2012-03-03", "0000764478-12-000035", None, 19_000_000, "764478/000076447812000035/bby-201210k.htm", "Restructuring charges in cost of goods sold"],
    ["BBY", "2016-01-30", "0000764478-16-000064", None, 3_000_000, "764478/000076447816000064/bby-2016x10k.htm", "Restructuring charges in cost of goods sold"],
    ["DDS", "2020-02-01", "0000028917-20-000041", 6_203_520_000, 0, "28917/000002891720000041/dds-02012020x10k.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2021-01-30", "0000028917-21-000066", 4_300_895_000, 0, "28917/000002891721000066/dds-20210130.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2022-01-29", "0000028917-22-000009", 6_492_993_000, 0, "28917/000002891722000009/dds-20220129.htm", "Net sales exclude service charges and other income"],
    ["DDS", "2023-01-28", "0000028917-23-000007", 6_871_081_000, 0, "28917/000002891723000007/R4.htm", "Net sales exclude service charges and other income"],
    ["URBN", "2020-01-31", "0001564590-20-014397", None, 14_611_000, "912615/000156459021017306/urbn-10k_20210131.htm", "Store impairment; comparative column in 2021 filing"],
    ["URBN", "2021-01-31", "0001564590-21-017306", None, 15_496_000, "912615/000156459021017306/urbn-10k_20210131.htm", "Store impairment"],
    ["URBN", "2023-01-31", "0000950170-23-011597", None, 6_417_000, "912615/000095017025048506/urbn-20250131.htm", "Store charges; comparative column in 2025 filing"],
    ["URBN", "2024-01-31", "0000950170-24-039320", None, 11_875_000, "912615/000095017025048506/urbn-20250131.htm", "Store charges; comparative column in 2025 filing"],
    ["URBN", "2025-01-31", "0000950170-25-048506", None, 4_601_000, "912615/000095017025048506/urbn-20250131.htm", "Store impairment and lease abandonment charges"],
    ["URBN", "2026-01-31", "0001193125-26-137916", None, 1_989_000, "912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf", "Store charges; matching annual report in a separate ARS filing"],
], columns=["ticker", "period_end", "accn", "reported_net_sales", "extra_gross_profit_charge", "evidence_url", "evidence_note"])
evidence["period_end"] = pd.to_datetime(evidence["period_end"])
evidence["evidence_url"] = sec_archive + evidence["evidence_url"]
annual = annual.merge(evidence, on=["ticker", "period_end", "accn"], how="left", validate="one_to_one")
annual["revenue_for_analysis"] = annual["reported_net_sales"].fillna(annual["revenue"])
annual["revenue_basis_adjusted"] = annual["reported_net_sales"].notna()
annual["remaining_bridge_gap"] = (
    annual["revenue_for_analysis"] - annual["cost_of_sales"]
    - annual["extra_gross_profit_charge"].fillna(0) - annual["gross_profit"]
)
annual["bridge_unresolved"] = annual["remaining_bridge_gap"].abs() > 1
matched = annual["evidence_url"].notna()
assert matched.sum() == len(evidence), "An accounting evidence row did not match the selected filing."
assert annual.loc[matched, "remaining_bridge_gap"].abs().le(1).all(), "Review the accounting reconciliation."
print("Evidence matches:", int(matched.sum()))
print("Revenue basis adjustments:", int(annual["revenue_basis_adjusted"].sum()))
print("Unresolved nonzero bridge gaps:", int(annual["bridge_unresolved"].sum()))
display(annual.loc[annual["revenue_basis_adjusted"], ["ticker", "period_end", "revenue", "revenue_for_analysis"]])

Evidence matches: 13
Revenue basis adjustments: 4
Unresolved nonzero bridge gaps: 0


    ticker period_end     revenue  revenue_for_analysis
194    DDS 2020-02-01  6343211000          6.203520e+09
195    DDS 2021-01-30  4433185000          4.300895e+09
196    DDS 2022-01-29  6624267000          6.492993e+09
197    DDS 2023-01-28  6996215000          6.871081e+09

In [118]:
annual["base_values_ok"] = (
    annual["revenue_for_analysis"].gt(0) & annual["total_assets"].gt(0)
    & annual["inventory"].ge(0) & annual["inventory"].le(annual["total_assets"])
)
annual["gm_row_usable"] = annual["base_values_ok"] & annual["gross_profit"].notna() & ~annual["bridge_unresolved"]
annual["ocf_row_usable"] = annual["base_values_ok"] & annual["operating_cash_flow"].notna()
annual["gross_margin"] = annual["gross_profit"] / annual["revenue_for_analysis"]
annual["ocf_margin"] = annual["operating_cash_flow"] / annual["revenue_for_analysis"]
annual.loc[~annual["gm_row_usable"], "gross_margin"] = float("nan")
annual.loc[~annual["ocf_row_usable"], "ocf_margin"] = float("nan")
annual["historical_scope_hold"] = annual["ticker"].eq("BBY") & annual["period_end"].lt(pd.Timestamp("2017-01-01"))
coverage = readiness.set_index("ticker")
company_allowed = annual["include_decision"].isin(["pending", "include"])
annual["gm_coverage_ok"] = annual["ticker"].map(coverage["gm_usable_years"]).ge(5) & company_allowed
annual["ocf_coverage_ok"] = annual["ticker"].map(coverage["ocf_usable_years"]).ge(5) & company_allowed
print("Historical rows held for further scope review:", int(annual["historical_scope_hold"].sum()))

Historical rows held for further scope review: 8


In [119]:
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
paired = annual.copy()
for column in [
    "period_end", "period_days", "revenue_for_analysis", "inventory",
    "base_values_ok", "known_scope_break", "historical_scope_hold",
]:
    paired["prev_" + column] = annual.groupby("ticker")[column].shift(1)
for column in [
    "period_start", "period_end", "period_days", "filed", "accn", "gross_margin", "ocf_margin",
    "gm_row_usable", "ocf_row_usable", "known_scope_break", "historical_scope_hold",
]:
    paired["next_" + column] = annual.groupby("ticker")[column].shift(-1)
display(paired[["ticker", "prev_period_end", "period_end", "next_period_end"]].head(8))

  ticker prev_period_end period_end next_period_end
0    AEO             NaT 2009-01-31      2010-01-30
1    AEO      2009-01-31 2010-01-30      2011-01-29
2    AEO      2010-01-30 2011-01-29      2012-01-28
3    AEO      2011-01-29 2012-01-28      2013-02-02
4    AEO      2012-01-28 2013-02-02      2014-02-01
5    AEO      2013-02-02 2014-02-01      2015-01-31
6    AEO      2014-02-01 2015-01-31      2016-01-30
7    AEO      2015-01-31 2016-01-30      2017-01-28

In [120]:
previous_end_gap = (paired["period_end"] - paired["prev_period_end"]).dt.days
previous_start_gap = (paired["period_start"] - paired["prev_period_end"]).dt.days
paired["consecutive_previous_year"] = previous_end_gap.between(350, 380) & previous_start_gap.eq(1)
paired["feature_usable"] = (
    paired["consecutive_previous_year"] & paired["base_values_ok"] & paired["prev_base_values_ok"].eq(True)
    & paired["prev_revenue_for_analysis"].gt(0) & paired["prev_inventory"].gt(0)
    & ~paired["known_scope_break"] & paired["prev_known_scope_break"].eq(False)
    & ~paired["historical_scope_hold"] & paired["prev_historical_scope_hold"].eq(False)
)
paired["revenue_growth"] = paired["revenue_for_analysis"] / paired["prev_revenue_for_analysis"] - 1
paired["inventory_growth"] = paired["inventory"] / paired["prev_inventory"] - 1
paired["inventory_sales_gap"] = paired["inventory_growth"] - paired["revenue_growth"]
paired.loc[~paired["feature_usable"], ["revenue_growth", "inventory_growth", "inventory_sales_gap"]] = float("nan")
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"].gt(0).astype("boolean")
paired.loc[~paired["feature_usable"], "inventory_outpaced_sales"] = pd.NA
next_end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
next_start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = next_end_gap.between(350, 380) & next_start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (paired["filed"] >= paired["period_end"]) & (paired["filed"] < paired["next_period_end"])
common_pair_ok = (
    paired["feature_usable"] & paired["consecutive_next_year"] & paired["filed_time_ok"]
    & paired["next_known_scope_break"].eq(False) & paired["next_historical_scope_hold"].eq(False)
)
paired["gm_pair_usable"] = common_pair_ok & paired["gm_coverage_ok"] & paired["gm_row_usable"] & paired["next_gm_row_usable"].eq(True)
paired["ocf_pair_usable"] = common_pair_ok & paired["ocf_coverage_ok"] & paired["ocf_row_usable"] & paired["next_ocf_row_usable"].eq(True)
print("Usable inventory feature rows:", int(paired["feature_usable"].sum()))

Usable inventory feature rows: 538


In [121]:
paired["next_gross_margin_change"] = paired["next_gross_margin"] - paired["gross_margin"]
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
for outcome, label, usable in [
    ("next_gross_margin_change", "gross_margin_declined", "gm_pair_usable"),
    ("next_ocf_margin_change", "ocf_margin_declined", "ocf_pair_usable"),
]:
    paired.loc[~paired[usable], outcome] = float("nan")
    paired[label] = paired[outcome].lt(0).astype("boolean")
    paired.loc[~paired[usable], label] = pd.NA
paired["available_before_target_start"] = paired["filed"] < paired["next_period_start"]
paired["days_after_target_start"] = (paired["filed"] - paired["next_period_start"]).dt.days
paired["period_length_changed"] = (paired["period_days"] - paired["prev_period_days"]).abs() > 3
paired["pair_period_length_changed"] = paired["period_length_changed"] | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
pair_summary = paired.groupby(["ticker", "subgroup"], sort=False)[["gm_pair_usable", "ocf_pair_usable"]].sum()
pair_summary = pair_summary.rename(columns={"gm_pair_usable": "gm_pairs", "ocf_pair_usable": "ocf_pairs"}).reset_index()
for name, table in [("Gross margin", gm_pairs), ("Operating cash flow", ocf_pairs)]:
    print(name, "companies:", table["ticker"].nunique(), "| pairs:", len(table))
    print("Available before target year starts:", int(table["available_before_target_start"].sum()))
    print("Pairs with different annual lengths:", int(table["pair_period_length_changed"].sum()))
    display(table["days_after_target_start"].agg(["min", "max"]).to_frame("Days After Target Start"))
display(pair_summary)
display(paired[["feature_usable", "consecutive_previous_year", "consecutive_next_year", "gm_pair_usable", "ocf_pair_usable"]].sum().rename("Rows Passing").to_frame())

Gross margin companies: 36 | pairs: 401
Available before target year starts: 0
Pairs with different annual lengths: 200


     Days After Target Start
min                     38.0
max                    134.0

Operating cash flow companies: 42 | pairs: 490
Available before target year starts: 0
Pairs with different annual lengths: 255


     Days After Target Start
min                     38.0
max                    134.0

   ticker                                           subgroup  gm_pairs  \
0     AEO            Apparel, Footwear, and Off-Price Retail        12   
1     AKA            Apparel, Footwear, and Off-Price Retail         4   
2     ANF            Apparel, Footwear, and Off-Price Retail        15   
3    ARHS  Home Furnishings, Furniture, and Home Improvement         4   
4     ASO  Sporting Goods, Electronics, and Other Merchan...         5   
5    BBWI                           Beauty and Personal Care        13   
6     BBY  Sporting Goods, Electronics, and Other Merchan...         8   
7     BKE            Apparel, Footwear, and Off-Price Retail        15   
8    BOOT            Apparel, Footwear, and Off-Price Retail        11   
9    BURL            Apparel, Footwear, and Off-Price Retail         0   
10   CATO            Apparel, Footwear, and Off-Price Retail         0   
11   CTRN            Apparel, Footwear, and Off-Price Retail         0   
12   CURV            Apparel, Footwear

                           Rows Passing
feature_usable                      538
consecutive_previous_year           612
consecutive_next_year               612
gm_pair_usable                      401
ocf_pair_usable                     490

In [122]:
outputs = {
    "expanded_module9_annual_audit.csv": annual,
    "expanded_module9_pairing_audit.csv": paired,
    "expanded_gm_next_year_pairs.csv": gm_pairs,
    "expanded_ocf_next_year_pairs.csv": ocf_pairs,
    "expanded_next_year_pair_counts.csv": pair_summary,
    "expanded_accounting_bridge_evidence.csv": evidence,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module8_metadata": module8_metadata,
    "annual_rows": len(annual), "accounting_evidence_rows": len(evidence),
    "revenue_basis_adjustments": int(annual["revenue_basis_adjusted"].sum()),
    "gm_companies": gm_pairs["ticker"].nunique(), "gm_pairs": len(gm_pairs),
    "ocf_companies": ocf_pairs["ticker"].nunique(), "ocf_pairs": len(ocf_pairs),
    "sample_status": "provisional", "min_annual_coverage": 5, "manual_metric_decisions_applied": False,
    "unit": "USD; ratios and growth rates stored as decimals",
    "revenue_rule": "Use verified net sales for the four identified DDS rows; preserve original revenue",
    "bridge_rule": "Preserve reported gross profit; record independently verified extra charges separately",
    "scope_rule": "Exclude features or targets involving known breaks; hold BBY period ends before 2017-01-01",
    "historical_scope_hold_sources": [
        "https://www.sec.gov/Archives/edgar/data/764478/000076447812000035/bby-201210k.htm",
        "https://www.sec.gov/Archives/edgar/data/764478/000076447816000064/bby-2016x10k.htm",
    ],
    "continuity_rule": "Adjacent annual ends 350-380 days apart and the next start one day after the prior end",
    "timing_rule": "Current filing date is a proxy; it must precede target end, but may follow target start",
    "research_design": "Retrospective annual association; current and target ratios both required for change outcomes",
    "accounting_scope_verified": False, "point_in_time_verified": False, "next_year_pairs_built": True,
}
(PROCESSED_DIR / "expanded_module9_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 9 outputs saved in:", PROCESSED_DIR)

Module 9 outputs saved in: [project root]/work/data/processed


## Module 12: Review Filing Evidence and Apply Sample Decisions

In [60]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
REVIEW_DIR = PROCESSED_DIR / "module12_manual_review"
MODULE12_OUTPUT_DIR = PROCESSED_DIR / "module12_reviewed"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
MODULE12_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_SAMPLE = "provisional"
MIN_ANNUAL_ROWS = 5
assert REPORT_SAMPLE in ["provisional", "reviewed_only"], "Unknown sample mode."
print("Project directory:", PROJECT_ROOT)
print("Review files:", REVIEW_DIR)
print("Sample mode:", REPORT_SAMPLE)

Project directory: [project root]
Review files: [project root]/work/data/processed/module12_manual_review
Sample mode: provisional


In [62]:
annual_source = pd.read_csv(PROCESSED_DIR / "expanded_module9_annual_audit.csv", parse_dates=["period_start", "period_end", "filed"])
annual_source = annual_source.sort_values(["ticker", "period_end"]).reset_index(drop=True)
company_source = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
metric_source = pd.read_csv(PROCESSED_DIR / "expanded_metric_review_decisions.csv", keep_default_na=False)
# Edited Version
case_source = pd.read_csv(
    PROCESSED_DIR / "expanded_module11_case_review.csv",
    dtype={"cik10": "string"},
    keep_default_na=False,
) 
company_source["eligibility_review_status"] = "pending"
company_source["eligibility_evidence_url"] = ""
company_source["eligibility_notes"] = ""
metric_source["reviewed_value_basis"] = ""
case_source["suggested_action"] = case_source["suggested_action"].replace("", "pending")
annual_review = annual_source[["ticker", "period_start", "period_end", "accn", "scope_reason", "known_scope_break", "historical_scope_hold"]].copy()
cik_lookup = company_source.set_index("ticker")["cik10"]
annual_review["current_filing_url"] = (
    "https://www.sec.gov/Archives/edgar/data/" + annual_review["ticker"].map(cik_lookup).astype(int).astype(str)
    + "/" + annual_review["accn"].str.replace("-", "", regex=False) + "/"
)
for direction, step in [("prev", 1), ("next", -1)]:
    annual_review[direction + "_period_end"] = annual_review.groupby("ticker")["period_end"].shift(step)
    annual_review[direction + "_filing_url"] = annual_review.groupby("ticker")["current_filing_url"].shift(step)
annual_review["related_to_selected_case"] = False
for _, case in case_source.iterrows():
    selected = annual_review["ticker"].eq(case["ticker"]) & annual_review["period_end"].eq(pd.Timestamp(case["period_end"]))
    dates = annual_review.loc[selected, ["prev_period_end", "period_end", "next_period_end"]].iloc[0]
    related = annual_review["ticker"].eq(case["ticker"]) & annual_review["period_end"].isin(dates)
    annual_review.loc[related, "related_to_selected_case"] = True
annual_review["scope_status"] = "pending"
annual_review["scope_evidence_url"] = ""
annual_review["scope_notes"] = ""
annual_review = annual_review.sort_values(["related_to_selected_case", "period_end"], ascending=[False, False])
tables = {"company_review.csv": company_source, "annual_scope_review.csv": annual_review, "metric_review.csv": metric_source, "case_review.csv": case_source}
for filename, table in tables.items():
    path = REVIEW_DIR / filename
    if not path.exists():
        table.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(filename, "| rows:", len(table), "| existing notes retained")
display(annual_review.loc[annual_review["related_to_selected_case"], ["ticker", "period_end", "current_filing_url"]])

company_review.csv | rows: 42 | existing notes retained
annual_scope_review.csv | rows: 656 | existing notes retained
metric_review.csv | rows: 150 | existing notes retained
case_review.csv | rows: 7 | existing notes retained


    ticker period_end                                 current_filing_url
547   SPWH 2023-01-28  https://www.sec.gov/Archives/edgar/data/113210...
21     AKA 2022-12-31  https://www.sec.gov/Archives/edgar/data/186510...
46    ARHS 2022-12-31  https://www.sec.gov/Archives/edgar/data/187544...
546   SPWH 2022-01-29  https://www.sec.gov/Archives/edgar/data/113210...
20     AKA 2021-12-31  https://www.sec.gov/Archives/edgar/data/186510...
45    ARHS 2021-12-31  https://www.sec.gov/Archives/edgar/data/187544...
545   SPWH 2021-01-30  https://www.sec.gov/Archives/edgar/data/113210...
19     AKA 2020-12-31  https://www.sec.gov/Archives/edgar/data/186510...
44    ARHS 2020-12-31  https://www.sec.gov/Archives/edgar/data/187544...
544   SPWH 2020-02-01  https://www.sec.gov/Archives/edgar/data/113210...
367    KSS 2014-02-01  https://www.sec.gov/Archives/edgar/data/885639...
590   ULTA 2014-02-01  https://www.sec.gov/Archives/edgar/data/140356...
366    KSS 2013-02-02  https://www.sec.gov/Archives

In [63]:
company_review = pd.read_csv(REVIEW_DIR / "company_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
scope_review = pd.read_csv(REVIEW_DIR / "annual_scope_review.csv", parse_dates=["period_end"], keep_default_na=False)
metric_review = pd.read_csv(REVIEW_DIR / "metric_review.csv", parse_dates=["period_end"], keep_default_na=False)
case_review = pd.read_csv(REVIEW_DIR / "case_review.csv", parse_dates=["period_end", "next_period_end"], keep_default_na=False)
assert company_review["eligibility_review_status"].isin(["pending", "reviewed"]).all(), "Invalid company review status."
assert company_review["include_decision"].isin(["pending", "include", "exclude"]).all(), "Invalid inclusion decision."
assert scope_review["scope_status"].isin(["pending", "reviewed", "exclude"]).all(), "Invalid annual scope status."
assert metric_review["review_status"].isin(["pending", "approved", "rejected"]).all(), "Invalid metric review status."
assert case_review["review_status"].isin(["pending", "reviewed"]).all(), "Invalid case review status."
assert case_review["suggested_action"].isin(["pending", "retain", "exclude_pair"]).all(), "Invalid case action."
assert len(company_review) == len(company_source), "Company list changed; reconcile review keys."
identity = company_review.merge(company_source[["ticker", "cik10", "subgroup"]], on=["ticker", "cik10", "subgroup"], validate="one_to_one", indicator=True)
assert len(identity) == len(company_source), "Company identity or group changed; reconcile review keys."
for review, source, keys in [
    (scope_review, annual_source, ["ticker", "period_end", "accn"]),
    (metric_review, metric_source.assign(period_end=pd.to_datetime(metric_source["period_end"])), ["ticker", "period_end", "accn", "metric"]),
    (case_review, case_source.assign(period_end=pd.to_datetime(case_source["period_end"]), next_period_end=pd.to_datetime(case_source["next_period_end"])), ["ticker", "period_end", "accn", "next_period_end", "next_accn"]),
]:
    matched = review.merge(source[keys], on=keys, how="left", validate="one_to_one", indicator=True)
    assert len(review) == len(source) and matched["_merge"].eq("both").all(), "Review keys changed; reconcile them before applying decisions."
for table, selected, columns in [
    (company_review, company_review["eligibility_review_status"].eq("reviewed") | company_review["include_decision"].ne("pending"), ["eligibility_evidence_url", "eligibility_notes"]),
    (scope_review, scope_review["scope_status"].ne("pending"), ["scope_evidence_url", "scope_notes"]),
    (metric_review, metric_review["review_status"].eq("approved"), ["evidence_url", "review_notes"]),
    (case_review, case_review["review_status"].eq("reviewed"), ["current_evidence_url", "next_evidence_url", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "review_notes"]),
]:
    for column in columns:
        assert table.loc[selected, column].str.strip().ne("").all(), "Reviewed decisions need evidence and notes: " + column
decided = company_review["include_decision"].ne("pending")
assert company_review.loc[decided, "eligibility_review_status"].eq("reviewed").all(), "Company decisions need a completed eligibility review."
assert company_review.loc[company_review["eligibility_review_status"].eq("reviewed"), "include_decision"].ne("pending").all(), "Reviewed companies need an inclusion decision."
assert case_review.loc[case_review["review_status"].eq("reviewed"), "suggested_action"].isin(["retain", "exclude_pair"]).all(), "Reviewed cases need an action."
assert case_review.loc[case_review["review_status"].eq("pending"), "suggested_action"].eq("pending").all(), "Case actions require completed evidence review."
approved = metric_review.loc[metric_review["review_status"].eq("approved")].copy()
approved["reviewed_value"] = pd.to_numeric(approved["reviewed_value"], errors="coerce")
assert np.isfinite(approved["reviewed_value"].to_numpy(dtype=float)).all(), "Approved values must be finite USD amounts."
assert approved["reviewed_value_basis"].isin(["reported", "derived"]).all(), "Approved values need a reported or derived basis."
assert approved.loc[approved["metric"].eq("cost_of_sales"), "reviewed_value"].ge(0).all(), "Cost of sales must be nonnegative."
print("Reviewed companies:", int(company_review["eligibility_review_status"].eq("reviewed").sum()))
print("Reviewed annual rows:", int(scope_review["scope_status"].eq("reviewed").sum()))
print("Approved metric values:", len(approved))
print("Reviewed cases:", int(case_review["review_status"].eq("reviewed").sum()))

Reviewed companies: 2
Reviewed annual rows: 7
Approved metric values: 0
Reviewed cases: 3


In [64]:
annual = annual_source.copy()
annual["original_cost_of_sales"] = annual["cost_of_sales"]
annual["original_gross_profit"] = annual["gross_profit"]
annual["manual_metrics_applied"] = ""
applied_rows = []
for _, decision in approved.iterrows():
    metric = decision["metric"]
    assert metric in ["cost_of_sales", "gross_profit"], "This module only fills missing cost or gross profit."
    match = annual["ticker"].eq(decision["ticker"]) & annual["period_end"].eq(decision["period_end"]) & annual["accn"].eq(decision["accn"])
    assert match.sum() == 1 and annual.loc[match, metric].isna().all(), "Approved values may only fill a matching missing field."
    annual.loc[match, metric] = decision["reviewed_value"]
    annual.loc[match, "manual_metrics_applied"] += metric + ";"
    if decision["reviewed_value_basis"] == "derived":
        annual.loc[match, "derived_fields"] += 1
    applied_rows.append(decision[["ticker", "period_end", "accn", "metric", "reviewed_value", "reviewed_value_basis", "evidence_url", "review_notes"]].to_dict())
applied_values = pd.DataFrame(applied_rows, columns=["ticker", "period_end", "accn", "metric", "reviewed_value", "reviewed_value_basis", "evidence_url", "review_notes"])
annual = annual.merge(scope_review[["ticker", "period_end", "accn", "scope_status", "scope_evidence_url", "scope_notes"]], on=["ticker", "period_end", "accn"], validate="one_to_one")
company_lookup = company_review.set_index("ticker")
annual["include_decision"] = annual["ticker"].map(company_lookup["include_decision"])
annual["eligibility_review_status"] = annual["ticker"].map(company_lookup["eligibility_review_status"])
company_allowed = annual["include_decision"].ne("exclude")
annual["row_allowed"] = annual["scope_status"].ne("exclude")
if REPORT_SAMPLE == "reviewed_only":
    company_allowed = annual["include_decision"].eq("include") & annual["eligibility_review_status"].eq("reviewed")
    annual["row_allowed"] = annual["scope_status"].eq("reviewed")
annual["remaining_bridge_gap"] = annual["revenue_for_analysis"] - annual["cost_of_sales"] - annual["extra_gross_profit_charge"].fillna(0) - annual["gross_profit"]
annual["bridge_unresolved"] = annual["remaining_bridge_gap"].abs() > 1
annual["numeric_base_values_ok"] = annual["revenue_for_analysis"].gt(0) & annual["total_assets"].gt(0) & annual["inventory"].ge(0) & annual["inventory"].le(annual["total_assets"])
annual["base_values_ok"] = annual["numeric_base_values_ok"] & annual["row_allowed"]
annual["gm_row_usable"] = annual["base_values_ok"] & annual["gross_profit"].notna() & ~annual["bridge_unresolved"]
annual["ocf_row_usable"] = annual["base_values_ok"] & annual["operating_cash_flow"].notna()
annual["gross_margin"] = annual["gross_profit"] / annual["revenue_for_analysis"]
annual["ocf_margin"] = annual["operating_cash_flow"] / annual["revenue_for_analysis"]
annual.loc[~annual["gm_row_usable"], "gross_margin"] = float("nan")
annual.loc[~annual["ocf_row_usable"], "ocf_margin"] = float("nan")
coverage = annual.groupby("ticker")[["gm_row_usable", "ocf_row_usable"]].sum()
annual["gm_coverage_ok"] = annual["ticker"].map(coverage["gm_row_usable"]).ge(MIN_ANNUAL_ROWS) & company_allowed
annual["ocf_coverage_ok"] = annual["ticker"].map(coverage["ocf_row_usable"]).ge(MIN_ANNUAL_ROWS) & company_allowed
print("Applied missing values:", len(applied_values))
print("Annual rows retained for the timeline:", len(annual))

Applied missing values: 0
Annual rows retained for the timeline: 656


In [65]:
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
paired = annual.copy()
for column in [
    "period_end", "period_days", "revenue_for_analysis", "inventory",
    "base_values_ok", "known_scope_break", "historical_scope_hold",
]:
    paired["prev_" + column] = annual.groupby("ticker")[column].shift(1)
for column in [
    "period_start", "period_end", "period_days", "filed", "accn", "gross_margin", "ocf_margin",
    "gm_row_usable", "ocf_row_usable", "known_scope_break", "historical_scope_hold",
]:
    paired["next_" + column] = annual.groupby("ticker")[column].shift(-1)
previous_end_gap = (paired["period_end"] - paired["prev_period_end"]).dt.days
previous_start_gap = (paired["period_start"] - paired["prev_period_end"]).dt.days
paired["consecutive_previous_year"] = previous_end_gap.between(350, 380) & previous_start_gap.eq(1)
paired["feature_usable"] = (
    paired["consecutive_previous_year"] & paired["base_values_ok"] & paired["prev_base_values_ok"].eq(True)
    & paired["prev_revenue_for_analysis"].gt(0) & paired["prev_inventory"].gt(0)
    & ~paired["known_scope_break"] & paired["prev_known_scope_break"].eq(False)
    & ~paired["historical_scope_hold"] & paired["prev_historical_scope_hold"].eq(False)
)
paired["revenue_growth"] = paired["revenue_for_analysis"] / paired["prev_revenue_for_analysis"] - 1
paired["inventory_growth"] = paired["inventory"] / paired["prev_inventory"] - 1
paired["inventory_sales_gap"] = paired["inventory_growth"] - paired["revenue_growth"]
paired.loc[~paired["feature_usable"], ["revenue_growth", "inventory_growth", "inventory_sales_gap"]] = float("nan")
paired["inventory_outpaced_sales"] = paired["inventory_sales_gap"].gt(0).astype("boolean")
paired.loc[~paired["feature_usable"], "inventory_outpaced_sales"] = pd.NA
next_end_gap = (paired["next_period_end"] - paired["period_end"]).dt.days
next_start_gap = (paired["next_period_start"] - paired["period_end"]).dt.days
paired["consecutive_next_year"] = next_end_gap.between(350, 380) & next_start_gap.eq(1)
paired["signal_available_date"] = paired["filed"]
paired["filed_time_ok"] = (paired["filed"] >= paired["period_end"]) & (paired["filed"] < paired["next_period_end"])
excluded_case_keys = case_review.loc[case_review["review_status"].eq("reviewed") & case_review["suggested_action"].eq("exclude_pair"), ["ticker", "period_end"]]
paired["case_pair_excluded"] = False
for _, key in excluded_case_keys.iterrows():
    match = paired["ticker"].eq(key["ticker"]) & paired["period_end"].eq(key["period_end"])
    paired.loc[match, "case_pair_excluded"] = True
common_pair_ok = (
    paired["feature_usable"] & paired["consecutive_next_year"] & paired["filed_time_ok"] & ~paired["case_pair_excluded"]
    & paired["next_known_scope_break"].eq(False) & paired["next_historical_scope_hold"].eq(False)
)
paired["gm_pair_usable"] = common_pair_ok & paired["gm_coverage_ok"] & paired["gm_row_usable"] & paired["next_gm_row_usable"].eq(True)
paired["ocf_pair_usable"] = common_pair_ok & paired["ocf_coverage_ok"] & paired["ocf_row_usable"] & paired["next_ocf_row_usable"].eq(True)
print("Usable inventory feature rows:", int(paired["feature_usable"].sum()))
paired["next_gross_margin_change"] = paired["next_gross_margin"] - paired["gross_margin"]
paired["next_ocf_margin_change"] = paired["next_ocf_margin"] - paired["ocf_margin"]
for outcome, label, usable in [
    ("next_gross_margin_change", "gross_margin_declined", "gm_pair_usable"),
    ("next_ocf_margin_change", "ocf_margin_declined", "ocf_pair_usable"),
]:
    paired.loc[~paired[usable], outcome] = float("nan")
    paired[label] = paired[outcome].lt(0).astype("boolean")
    paired.loc[~paired[usable], label] = pd.NA
paired["available_before_target_start"] = paired["filed"] < paired["next_period_start"]
paired["days_after_target_start"] = (paired["filed"] - paired["next_period_start"]).dt.days
paired["period_length_changed"] = (paired["period_days"] - paired["prev_period_days"]).abs() > 3
paired["pair_period_length_changed"] = paired["period_length_changed"] | ((paired["next_period_days"] - paired["period_days"]).abs() > 3)
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
pair_summary = paired.groupby(["ticker", "subgroup"], sort=False)[["gm_pair_usable", "ocf_pair_usable"]].sum()
pair_summary = pair_summary.rename(columns={"gm_pair_usable": "gm_pairs", "ocf_pair_usable": "ocf_pairs"}).reset_index()

print("Rebuilt gross margin pairs:", len(gm_pairs))
print("Rebuilt OCF-to-revenue pairs:", len(ocf_pairs))

Usable inventory feature rows: 538
Rebuilt gross margin pairs: 401
Rebuilt OCF-to-revenue pairs: 490


In [66]:
changes = []
for name, filename, table in [
    ("Gross margin", "expanded_gm_next_year_pairs.csv", gm_pairs),
    ("OCF-to-revenue", "expanded_ocf_next_year_pairs.csv", ocf_pairs),
]:
    before = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end"])
    before_keys = before[["ticker", "period_end"]]
    after_keys = table[["ticker", "period_end"]]
    removed = before_keys.merge(after_keys, on=["ticker", "period_end"], how="left", indicator=True)
    added = after_keys.merge(before_keys, on=["ticker", "period_end"], how="left", indicator=True)
    changes.append({"outcome": name, "before_pairs": len(before), "after_pairs": len(table), "after_companies": table["ticker"].nunique(), "removed_pairs": int(removed["_merge"].eq("left_only").sum()), "added_pairs": int(added["_merge"].eq("left_only").sum())})
sample_changes = pd.DataFrame(changes)
case_columns = ["ticker", "period_end", "selection_reason", "review_status", "suggested_action", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "current_evidence_url", "next_evidence_url", "review_notes"]
case_results = case_review[case_columns].merge(paired, on=["ticker", "period_end"], validate="one_to_one")
case_results["inventory_gap_pp"] = case_results["inventory_sales_gap"] * 100
display(sample_changes)
display(case_results[["ticker", "period_end", "review_status", "suggested_action", "inventory_gap_pp", "gm_pair_usable", "ocf_pair_usable", "next_gross_margin_change", "next_ocf_margin_change"]])

          outcome  before_pairs  after_pairs  after_companies  removed_pairs  \
0    Gross margin           401          401               36              0   
1  OCF-to-revenue           490          490               42              0   

   added_pairs  
0            0  
1            0  

  ticker period_end review_status suggested_action  inventory_gap_pp  \
0    AKA 2021-12-31       pending          pending         89.169269   
1   SPWH 2022-01-29      reviewed           retain         55.053967   
2   ARHS 2021-12-31       pending          pending         35.819959   
3   ULTA 2013-02-02       pending          pending         22.606853   
4    KSS 2013-02-02       pending          pending         14.635555   
5   URBN 2011-01-31      reviewed           retain          5.979765   
6   SPWH 2021-01-30      reviewed           retain        -75.422998   

   gm_pair_usable  ocf_pair_usable  next_gross_margin_change  \
0            True             True                  0.004034   
1            True             True                  0.003310   
2            True             True                  0.013240   
3            True             True                 -0.000514   
4            True             True                  0.002308   
5            True             True     

In [67]:
paired["company_eligibility_verified"] = paired["eligibility_review_status"].eq("reviewed") & paired["include_decision"].eq("include")
paired["three_year_scope_reviewed"] = (
    paired["scope_status"].eq("reviewed")
    & annual.groupby("ticker")["scope_status"].shift(1).eq("reviewed")
    & annual.groupby("ticker")["scope_status"].shift(-1).eq("reviewed")
)
paired["review_complete_pair"] = paired["company_eligibility_verified"] & paired["three_year_scope_reviewed"]
gm_pairs = paired.loc[paired["gm_pair_usable"]].copy()
ocf_pairs = paired.loc[paired["ocf_pair_usable"]].copy()
readiness_rows = []
for name, table in [("Gross margin", gm_pairs), ("OCF-to-revenue", ocf_pairs)]:
    readiness_rows.append({"outcome": name, "pairs": len(table), "companies": table["ticker"].nunique(), "review_complete_pairs": int(table["review_complete_pair"].sum()), "pending_review_pairs": int((~table["review_complete_pair"]).sum()), "before_target_start_pairs": int(table["available_before_target_start"].sum()), "different_length_pairs": int(table["pair_period_length_changed"].sum())})
report_readiness = pd.DataFrame(readiness_rows)
display(report_readiness)
display(annual[["cost_of_sales", "gross_profit"]].isna().sum().rename("Remaining Missing Annual Fields").to_frame())
print("Point-in-time verified:", False)
print("Prediction validated:", False)
if gm_pairs.empty or ocf_pairs.empty:
    print("At least one outcome sample is empty. Continue evidence review before fitting that outcome.")

          outcome  pairs  companies  review_complete_pairs  \
0    Gross margin    401         36                      3   
1  OCF-to-revenue    490         42                      3   

   pending_review_pairs  before_target_start_pairs  different_length_pairs  
0                   398                          0                     200  
1                   487                          0                     255  

               Remaining Missing Annual Fields
cost_of_sales                               42
gross_profit                               108

Point-in-time verified: False
Prediction validated: False


In [68]:
metadata_path = PROJECT_ROOT / "work" / "data" / "processed" / "expanded_module9_metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
outputs = {
    "expanded_module12_annual_audit.csv": annual,
    "expanded_module12_pairing_audit.csv": paired,
    "expanded_gm_next_year_pairs.csv": gm_pairs,
    "expanded_ocf_next_year_pairs.csv": ocf_pairs,
    "expanded_next_year_pair_counts.csv": pair_summary,
    "expanded_module12_applied_values.csv": applied_values,
    "expanded_module12_sample_changes.csv": sample_changes,
    "expanded_module12_case_results.csv": case_results,
    "expanded_module12_report_readiness.csv": report_readiness,
    "candidate_scope_review.csv": company_review,
}
for filename, table in outputs.items():
    table.to_csv(MODULE12_OUTPUT_DIR / filename, index=False, date_format="%Y-%m-%d")
retained_keys = pd.concat([gm_pairs, ocf_pairs]).drop_duplicates(["ticker", "period_end"])
scope_verified = not retained_keys.empty and retained_keys["review_complete_pair"].all()
metadata.update({
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "review_module": 12,
    "report_sample_mode": REPORT_SAMPLE, "sample_status": "reviewed_retained_pairs" if scope_verified else "post_review_provisional",
    "gm_companies": gm_pairs["ticker"].nunique(), "gm_pairs": len(gm_pairs), "ocf_companies": ocf_pairs["ticker"].nunique(), "ocf_pairs": len(ocf_pairs),
    "min_annual_coverage": MIN_ANNUAL_ROWS, "coverage_rule": "Recount usable annual rows after numeric and annual-scope decisions; company eligibility also required in reviewed-only mode",
    "manual_metric_decisions_applied": not applied_values.empty,
    "manual_review_decisions_applied": bool(company_review["eligibility_review_status"].eq("reviewed").any() or scope_review["scope_status"].ne("pending").any() or case_review["review_status"].eq("reviewed").any() or not applied_values.empty),
    "applied_metric_values": len(applied_values), "reviewed_annual_rows": int(scope_review["scope_status"].eq("reviewed").sum()), "reviewed_cases": int(case_review["review_status"].eq("reviewed").sum()),
    "accounting_scope_verified": bool(scope_verified), "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False,
    "scope_rule": "Retain original known breaks and BBY hold; apply manual annual exclusions to all dependencies and documented case exclusions to individual pairs",
    "metric_rule": "Only approved missing cost/gross-profit values with recorded evidence are filled; original amounts preserved",
    "case_rule": "Keep Module 11 selected cases fixed; refresh computed results after review decisions",
    "review_directory": str(REVIEW_DIR),
})
for filename in ["expanded_module12_metadata.json", "expanded_module9_metadata.json"]:
    (MODULE12_OUTPUT_DIR / filename).write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 12 outputs saved in:", MODULE12_OUTPUT_DIR)
print("Retained-pair accounting review complete:", bool(scope_verified))

Module 12 outputs saved in: [project root]/work/data/processed/module12_reviewed
Retained-pair accounting review complete: False


## Module 10: Explore Expanded Inventory Signals and Check Sensitivity

In [69]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.size"] = 10
print("Project directory:", PROJECT_ROOT)

# Edited Version
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed" / "module12_reviewed"
FIGURE_DIR = PROCESSED_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

Project directory: [project root]


In [70]:
settings = {
    "Gross margin": ("expanded_gm_next_year_pairs.csv", "gross_margin", "next_gross_margin_change", "gross_margin_declined"),
    "OCF-to-revenue": ("expanded_ocf_next_year_pairs.csv", "ocf_margin", "next_ocf_margin_change", "ocf_margin_declined"),
}
analysis_tables = {}
for name, (filename, ratio, outcome, label) in settings.items():
    table = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end", "next_period_end"], dtype={label: "boolean"})
    assert not table.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year pairs."
    assert table[["inventory_sales_gap", outcome, label]].notna().all().all(), "Review missing signals or outcomes."
    table["inventory_gap_pp"] = table["inventory_sales_gap"] * 100
    table["outcome_change_pp"] = table[outcome] * 100
    table["current_ratio_pct"] = table[ratio] * 100
    table["next_ratio_pct"] = table["next_" + ratio] * 100
    table["declined"] = table[label]
    table["inventory_group"] = "Gap <= 0"
    table.loc[table["inventory_sales_gap"].gt(0), "inventory_group"] = "Gap > 0"
    analysis_tables[name] = table
    print(name, "pairs:", len(table), "| companies:", table["ticker"].nunique())
common_keys = analysis_tables["Gross margin"][["ticker", "period_end"]].merge(
    analysis_tables["OCF-to-revenue"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one",
)
module9_metadata = json.loads((PROCESSED_DIR / "expanded_module9_metadata.json").read_text(encoding="utf-8"))
first_end = min(table["period_end"].min() for table in analysis_tables.values()).date()
last_end = max(table["period_end"].max() for table in analysis_tables.values()).date()
source_caption = f"Source: cached SEC Company Facts / Module 9 | Current ends: {first_end} to {last_end} | Provisional"
print("Common outcome pairs:", len(common_keys))

Gross margin pairs: 401 | companies: 36
OCF-to-revenue pairs: 490 | companies: 42
Common outcome pairs: 399


In [71]:
coverage_tables = []
pooled_tables = []
for name, table in analysis_tables.items():
    coverage = table.groupby(["ticker", "subgroup"]).agg(
        pairs=("ticker", "size"), first_period_end=("period_end", "min"), last_period_end=("period_end", "max"),
    ).reset_index()
    coverage["outcome"] = name
    coverage_tables.append(coverage)
    summary = table.groupby("inventory_group").agg(
        pairs=("ticker", "size"), companies=("ticker", "nunique"),
        mean_change_pp=("outcome_change_pp", "mean"), median_change_pp=("outcome_change_pp", "median"),
        decline_rate=("declined", "mean"),
    ).reset_index()
    summary["outcome"] = name
    pooled_tables.append(summary)
    print(name, "signal and outcome distributions:")
    display(table[["inventory_gap_pp", "outcome_change_pp"]].describe().round(3))
sample_coverage = pd.concat(coverage_tables, ignore_index=True)
pooled_summary = pd.concat(pooled_tables, ignore_index=True)
display(pooled_summary.round(3))
display(sample_coverage.head(10))

Gross margin signal and outcome distributions:


       inventory_gap_pp  outcome_change_pp
count           401.000            401.000
mean              1.599              0.138
std              14.481              2.527
min             -75.423            -10.198
25%              -3.980             -0.697
50%               1.680             -0.058
75%               6.941              0.841
max              89.169             16.589

OCF-to-revenue signal and outcome distributions:


       inventory_gap_pp  outcome_change_pp
count           490.000            490.000
mean              1.123             -0.045
std              13.839              4.557
min             -75.423            -17.886
25%              -4.491             -2.117
50%               1.061             -0.226
75%               6.585              1.798
max              89.169             20.973

  inventory_group  pairs  companies  mean_change_pp  median_change_pp  \
0        Gap <= 0    170         36           0.164             0.071   
1         Gap > 0    231         36           0.120            -0.114   
2        Gap <= 0    219         42          -1.276            -0.856   
3         Gap > 0    271         42           0.949             0.419   

   decline_rate         outcome  
0         0.471    Gross margin  
1         0.571    Gross margin  
2         0.703  OCF-to-revenue  
3         0.417  OCF-to-revenue  

  ticker                                           subgroup  pairs  \
0    AEO            Apparel, Footwear, and Off-Price Retail     12   
1    AKA            Apparel, Footwear, and Off-Price Retail      4   
2    ANF            Apparel, Footwear, and Off-Price Retail     15   
3   ARHS  Home Furnishings, Furniture, and Home Improvement      4   
4    ASO  Sporting Goods, Electronics, and Other Merchan...      5   
5   BBWI                           Beauty and Personal Care     13   
6    BBY  Sporting Goods, Electronics, and Other Merchan...      8   
7    BKE            Apparel, Footwear, and Off-Price Retail     15   
8   BOOT            Apparel, Footwear, and Off-Price Retail     11   
9   CURV            Apparel, Footwear, and Off-Price Retail      4   

  first_period_end last_period_end       outcome  
0       2011-01-29      2025-02-01  Gross margin  
1       2021-12-31      2024-12-31  Gross margin  
2       2011-01-29      2025-02-01  Gross margin  
3       2021-12-31      2

In [72]:
short_groups = {
    "Apparel, Footwear, and Off-Price Retail": "Apparel / footwear / off-price",
    "Department Stores": "Department stores",
    "Home Furnishings, Furniture, and Home Improvement": "Home / furniture / improvement",
    "Sporting Goods, Electronics, and Other Merchandise": "Sporting / electronics / other",
    "Beauty and Personal Care": "Beauty / personal care",
}
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, (name, table) in zip(axes, analysis_tables.items()):
    for group_index, (subgroup, short_name) in enumerate(short_groups.items()):
        group = table.loc[table["subgroup"].eq(subgroup)]
        ax.scatter(group["inventory_gap_pp"], group["outcome_change_pp"], color=f"C{group_index}", label=short_name, s=25, alpha=0.65)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Inventory growth minus sales growth (pp)")
    ax.set_ylabel("Next-year ratio change (pp)")
    ax.set_title(f"{name}: {table['ticker'].nunique()} companies, {len(table)} pairs")
    ax.text(0.02, 0.98, f"Current ends: {table['period_end'].min().date()} to {table['period_end'].max().date()}", transform=ax.transAxes, va="top", fontsize=8)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.5, 0.045), ncol=3, fontsize=8)
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.18, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_inventory_gap_scatter.png", dpi=180, bbox_inches="tight")
plt.show()

<Figure size 1400x650 with 2 Axes>

In [73]:
subgroup_tables = []
difference_tables = []
for name, table in analysis_tables.items():
    summary = table.groupby(["subgroup", "inventory_group"]).agg(
        pairs=("ticker", "size"), companies=("ticker", "nunique"),
        mean_change_pp=("outcome_change_pp", "mean"), median_change_pp=("outcome_change_pp", "median"),
        decline_rate=("declined", "mean"),
    )
    means = summary["mean_change_pp"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    counts = summary["pairs"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    differences = pd.DataFrame({
        "mean_difference_pp": means["Gap > 0"] - means["Gap <= 0"],
        "positive_gap_pairs": counts["Gap > 0"], "nonpositive_gap_pairs": counts["Gap <= 0"],
    }).reset_index()
    summary = summary.reset_index()
    summary["outcome"] = name
    differences["outcome"] = name
    subgroup_tables.append(summary)
    difference_tables.append(differences)
subgroup_summary = pd.concat(subgroup_tables, ignore_index=True)
subgroup_differences = pd.concat(difference_tables, ignore_index=True)
display(subgroup_summary.round(3))
display(subgroup_differences.round(3))
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
for ax, name in zip(axes, analysis_tables):
    chart = subgroup_differences.loc[subgroup_differences["outcome"].eq(name)].dropna(subset=["mean_difference_pp"])
    labels = [f"{short_groups[row.subgroup]} (+:{int(row.positive_gap_pairs)}, -:{int(row.nonpositive_gap_pairs)})" for row in chart.itertuples()]
    bars = ax.barh(labels, chart["mean_difference_pp"], color="tab:blue", alpha=0.8)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Mean next-year change: Gap > 0 minus Gap <= 0 (pp)")
    ax.set_ylabel("Retail subgroup")
    ax.margins(x=0.2)
fig.text(0.5, 0.01, source_caption + "\n+ / -: positive / nonpositive gap pair counts", ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_subgroup_mean_differences.png", dpi=180, bbox_inches="tight")
plt.show()

                                             subgroup inventory_group  pairs  \
0             Apparel, Footwear, and Off-Price Retail        Gap <= 0     82   
1             Apparel, Footwear, and Off-Price Retail         Gap > 0    109   
2                            Beauty and Personal Care        Gap <= 0     20   
3                            Beauty and Personal Care         Gap > 0     22   
4                                   Department Stores        Gap <= 0     10   
5                                   Department Stores         Gap > 0     12   
6   Home Furnishings, Furniture, and Home Improvement        Gap <= 0     32   
7   Home Furnishings, Furniture, and Home Improvement         Gap > 0     50   
8   Sporting Goods, Electronics, and Other Merchan...        Gap <= 0     26   
9   Sporting Goods, Electronics, and Other Merchan...         Gap > 0     38   
10            Apparel, Footwear, and Off-Price Retail        Gap <= 0    122   
11            Apparel, Footwear, and Off

                                            subgroup  mean_difference_pp  \
0            Apparel, Footwear, and Off-Price Retail               0.042   
1                           Beauty and Personal Care               0.351   
2                                  Department Stores               1.252   
3  Home Furnishings, Furniture, and Home Improvement              -0.790   
4  Sporting Goods, Electronics, and Other Merchan...              -0.104   
5            Apparel, Footwear, and Off-Price Retail               1.997   
6                           Beauty and Personal Care               1.864   
7                                  Department Stores               3.301   
8  Home Furnishings, Furniture, and Home Improvement               2.260   
9  Sporting Goods, Electronics, and Other Merchan...               2.649   

   positive_gap_pairs  nonpositive_gap_pairs         outcome  
0                 109                     82    Gross margin  
1                  22                

<Figure size 1100x800 with 2 Axes>

In [74]:
MIN_GROUP_PAIRS = 3
company_tables = []
balanced_rows = []
for name, table in analysis_tables.items():
    grouped = table.groupby(["ticker", "subgroup", "inventory_group"])["outcome_change_pp"].agg(["size", "mean"])
    means = grouped["mean"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    counts = grouped["size"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    comparison = pd.DataFrame({
        "mean_difference_pp": means["Gap > 0"] - means["Gap <= 0"],
        "positive_gap_pairs": counts["Gap > 0"], "nonpositive_gap_pairs": counts["Gap <= 0"],
    }).reset_index()
    comparison["comparison_usable"] = comparison["positive_gap_pairs"].ge(MIN_GROUP_PAIRS) & comparison["nonpositive_gap_pairs"].ge(MIN_GROUP_PAIRS)
    comparison["outcome"] = name
    company_tables.append(comparison)
    valid = comparison.loc[comparison["comparison_usable"], "mean_difference_pp"]
    balanced_rows.append({
        "outcome": name, "companies": len(valid), "mean_company_difference_pp": valid.mean(),
        "median_company_difference_pp": valid.median(),
        "share_negative": valid.lt(0).mean() if len(valid) else float("nan"),
    })
company_comparisons = pd.concat(company_tables, ignore_index=True)
company_balanced_summary = pd.DataFrame(balanced_rows)
display(company_balanced_summary.round(3))
display(company_comparisons.head(10).round(3))

          outcome  companies  mean_company_difference_pp  \
0    Gross margin         31                       0.029   
1  OCF-to-revenue         36                       2.712   

   median_company_difference_pp  share_negative  
0                        -0.118           0.581  
1                         2.599           0.139  

  ticker                                           subgroup  \
0    AEO            Apparel, Footwear, and Off-Price Retail   
1    AKA            Apparel, Footwear, and Off-Price Retail   
2    ANF            Apparel, Footwear, and Off-Price Retail   
3   ARHS  Home Furnishings, Furniture, and Home Improvement   
4    ASO  Sporting Goods, Electronics, and Other Merchan...   
5   BBWI                           Beauty and Personal Care   
6    BBY  Sporting Goods, Electronics, and Other Merchan...   
7    BKE            Apparel, Footwear, and Off-Price Retail   
8   BOOT            Apparel, Footwear, and Off-Price Retail   
9   CURV            Apparel, Footwear, and Off-Price Retail   

   mean_difference_pp  positive_gap_pairs  nonpositive_gap_pairs  \
0               3.229                   8                      4   
1              -1.054                   2                      2   
2              -2.118                   9                      6   
3               2.038             

In [75]:
sensitivity_rows = []
trim_rows = []
for name, table in analysis_tables.items():
    lower = table["inventory_gap_pp"].quantile(0.01)
    upper = table["inventory_gap_pp"].quantile(0.99)
    trim_rows.append({"outcome": name, "lower_gap_pp": lower, "upper_gap_pp": upper})
    common = table.merge(common_keys, on=["ticker", "period_end"], validate="one_to_one")
    for sample_name, sample in [
        ("All pairs", table), ("Unchanged period lengths", table.loc[~table["pair_period_length_changed"]]),
        ("Common outcome pairs", common), ("Central 98% of inventory gaps", table.loc[table["inventory_gap_pp"].between(lower, upper)]),
    ]:
        x = sample["inventory_gap_pp"]
        y = sample["outcome_change_pp"]
        row = {"outcome": name, "sample": sample_name, "pairs": len(sample), "companies": sample["ticker"].nunique()}
        for column, result in [("outcome_change_pp", "pearson_change_r"), ("current_ratio_pct", "current_level_r"), ("next_ratio_pct", "next_level_r")]:
            row[result] = x.corr(sample[column]) if x.nunique() > 1 and sample[column].nunique() > 1 else float("nan")
        row["spearman_change_r"] = x.rank().corr(y.rank()) if x.nunique() > 1 and y.nunique() > 1 else float("nan")
        x_within = x - sample.groupby("ticker")["inventory_gap_pp"].transform("mean")
        y_within = y - sample.groupby("ticker")["outcome_change_pp"].transform("mean")
        row["company_demeaned_r"] = x_within.corr(y_within) if x_within.nunique() > 1 and y_within.nunique() > 1 else float("nan")
        sensitivity_rows.append(row)
sensitivity = pd.DataFrame(sensitivity_rows)
trim_limits = pd.DataFrame(trim_rows)
display(sensitivity.round(3))
display(trim_limits.round(3))

          outcome                         sample  pairs  companies  \
0    Gross margin                      All pairs    401         36   
1    Gross margin       Unchanged period lengths    201         36   
2    Gross margin           Common outcome pairs    399         36   
3    Gross margin  Central 98% of inventory gaps    393         36   
4  OCF-to-revenue                      All pairs    490         42   
5  OCF-to-revenue       Unchanged period lengths    235         42   
6  OCF-to-revenue           Common outcome pairs    399         36   
7  OCF-to-revenue  Central 98% of inventory gaps    480         42   

   pearson_change_r  current_level_r  next_level_r  spearman_change_r  \
0            -0.052           -0.006        -0.020             -0.108   
1             0.012           -0.002         0.001              0.006   
2            -0.053           -0.009        -0.022             -0.109   
3            -0.034           -0.042        -0.051             -0.100   
4   

          outcome  lower_gap_pp  upper_gap_pp
0    Gross margin       -42.620        42.988
1  OCF-to-revenue       -40.823        42.112

In [76]:
outputs = {
    "expanded_sample_coverage.csv": sample_coverage,
    "expanded_inventory_group_summary.csv": pooled_summary,
    "expanded_subgroup_group_summary.csv": subgroup_summary,
    "expanded_subgroup_mean_differences.csv": subgroup_differences,
    "expanded_company_group_comparisons.csv": company_comparisons,
    "expanded_company_balanced_summary.csv": company_balanced_summary,
    "expanded_correlation_sensitivity.csv": sensitivity,
    "expanded_inventory_trim_limits.csv": trim_limits,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module9_metadata": module9_metadata,
    "sample_status": "provisional", "common_outcome_pairs": len(common_keys),
    "inventory_group_rule": "Gap > 0 versus Gap <= 0; zero belongs to the nonpositive group",
    "within_company_min_pairs_per_group": MIN_GROUP_PAIRS,
    "weighting": "Pooled results weight annual pairs equally; balanced results weight eligible companies equally",
    "sensitivity_rules": ["All pairs", "Unchanged period lengths", "Common outcome pairs", "Central 98% of inventory gaps"],
    "trim_rule": "Outcome-specific 1st-99th inventory-gap percentiles, inclusive; original data retained",
    "correlation_design": "Descriptive Pearson, rank correlation and company-demeaned Pearson; no year adjustment",
    "unit": "Changes in percentage points; ratio levels in percent; rates and correlations in decimals",
    "p_values_computed": False, "causal_identification": False, "prediction_validated": False,
    "accounting_scope_verified": False, "point_in_time_verified": False,
}
(PROCESSED_DIR / "expanded_module10_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 10 tables saved in:", PROCESSED_DIR)
print("Module 10 figures saved in:", FIGURE_DIR)

Module 10 tables saved in: [project root]/work/data/processed/module12_reviewed
Module 10 figures saved in: [project root]/work/data/processed/module12_reviewed/figures


## Module 11: Control for Baseline Ratios and Prepare Filing Review Cases

In [77]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.formula.api as smf
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)
print("Statsmodels version:", statsmodels.__version__)

# Edited Version
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed" / "module12_reviewed"
FIGURE_DIR = PROCESSED_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

Project directory: [project root]
Statsmodels version: 0.14.6


In [78]:
settings = {
    "Gross margin": ("expanded_gm_next_year_pairs.csv", "gross_margin", "next_gross_margin_change"),
    "OCF-to-revenue": ("expanded_ocf_next_year_pairs.csv", "ocf_margin", "next_ocf_margin_change"),
}
regression_tables = {}
for name, (filename, ratio, outcome) in settings.items():
    table = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end", "next_period_end"])
    table["inventory_gap_pp"] = table["inventory_sales_gap"] * 100
    table["current_ratio_pct"] = table[ratio] * 100
    table["next_ratio_pct"] = table["next_" + ratio] * 100
    table["outcome_change_pp"] = table[outcome] * 100
    table["revenue_growth_pp"] = table["revenue_growth"] * 100
    table["log_assets"] = np.log(table["total_assets"])
    table["target_end_year"] = table["next_period_end"].dt.year
    inputs = ["inventory_gap_pp", "current_ratio_pct", "next_ratio_pct", "outcome_change_pp", "revenue_growth_pp", "log_assets", "target_end_year"]
    assert np.isfinite(table[inputs].to_numpy()).all(), "Review missing or invalid regression inputs."
    assert not table.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year pairs."
    regression_tables[name] = table
    print(name, "pairs:", len(table), "| companies:", table["ticker"].nunique())
common_keys = regression_tables["Gross margin"][["ticker", "period_end"]].merge(
    regression_tables["OCF-to-revenue"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one",
)
company_review = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
module10_metadata = json.loads((PROCESSED_DIR / "expanded_module10_metadata.json").read_text(encoding="utf-8"))
print("Common outcome pairs:", len(common_keys))

Gross margin pairs: 401 | companies: 36
OCF-to-revenue pairs: 490 | companies: 42
Common outcome pairs: 399


In [79]:
model_specs = {
    "Change: unadjusted": "outcome_change_pp ~ inventory_gap_pp",
    "Next level: unadjusted": "next_ratio_pct ~ inventory_gap_pp",
    "Next level: baseline": "next_ratio_pct ~ inventory_gap_pp + current_ratio_pct",
    "Change: baseline": "outcome_change_pp ~ inventory_gap_pp + current_ratio_pct",
    "Next level: company/year": "next_ratio_pct ~ inventory_gap_pp + current_ratio_pct + revenue_growth_pp + log_assets + C(ticker) + C(target_end_year)",
}
models = {}
for name, table in regression_tables.items():
    for model_name, formula in model_specs.items():
        model = smf.ols(formula, data=table, eval_env=-1).fit(
            cov_type="cluster", cov_kwds={"groups": table["ticker"], "use_correction": True, "df_correction": True}, use_t=True,
        )
        models[(name, model_name)] = model
print("Main model fits:", len(models))

Main model fits: 10


In [80]:
model_rows = []
for (name, model_name), model in models.items():
    interval = model.conf_int().loc["inventory_gap_pp"]
    model_rows.append({
        "outcome": name, "model": model_name, "pairs": int(model.nobs), "companies": regression_tables[name]["ticker"].nunique(),
        "parameters": len(model.params), "gap_coefficient": model.params["inventory_gap_pp"],
        "association_10pp": model.params["inventory_gap_pp"] * 10,
        "ci_lower_10pp": interval.iloc[0] * 10, "ci_upper_10pp": interval.iloc[1] * 10,
        "baseline_coefficient": model.params.get("current_ratio_pct", float("nan")), "r_squared": model.rsquared,
    })
model_results = pd.DataFrame(model_rows)
identity_rows = []
for name in regression_tables:
    level_model = models[(name, "Next level: baseline")]
    change_model = models[(name, "Change: baseline")]
    identity_rows.append({
        "outcome": name,
        "gap_coefficient_difference": change_model.params["inventory_gap_pp"] - level_model.params["inventory_gap_pp"],
        "baseline_difference_plus_one": change_model.params["current_ratio_pct"] - level_model.params["current_ratio_pct"] + 1,
    })
identity_check = pd.DataFrame(identity_rows)
display(model_results.round(4))
display(identity_check.round(10))

          outcome                     model  pairs  companies  parameters  \
0    Gross margin        Change: unadjusted    401         36           2   
1    Gross margin    Next level: unadjusted    401         36           2   
2    Gross margin      Next level: baseline    401         36           3   
3    Gross margin          Change: baseline    401         36           3   
4    Gross margin  Next level: company/year    401         36          55   
5  OCF-to-revenue        Change: unadjusted    490         42           2   
6  OCF-to-revenue    Next level: unadjusted    490         42           2   
7  OCF-to-revenue      Next level: baseline    490         42           3   
8  OCF-to-revenue          Change: baseline    490         42           3   
9  OCF-to-revenue  Next level: company/year    490         42          61   

   gap_coefficient  association_10pp  ci_lower_10pp  ci_upper_10pp  \
0          -0.0091           -0.0912        -0.2886         0.1062   
1          -

          outcome  gap_coefficient_difference  baseline_difference_plus_one
0    Gross margin                         0.0                           0.0
1  OCF-to-revenue                        -0.0                           0.0

In [81]:
sensitivity_rows = []
adjusted_formula = model_specs["Next level: company/year"]
for name, table in regression_tables.items():
    lower, upper = table["inventory_gap_pp"].quantile([0.01, 0.99])
    common = table.merge(common_keys, on=["ticker", "period_end"], validate="one_to_one")
    for sample_name, sample in [
        ("All pairs", table), ("Unchanged period lengths", table.loc[~table["pair_period_length_changed"]]),
        ("Common outcome pairs", common), ("Central 98% of inventory gaps", table.loc[table["inventory_gap_pp"].between(lower, upper)]),
    ]:
        if sample_name == "All pairs":
            model = models[(name, "Next level: company/year")]
        else:
            model = smf.ols(adjusted_formula, data=sample, eval_env=-1).fit(
                cov_type="cluster", cov_kwds={"groups": sample["ticker"], "use_correction": True, "df_correction": True}, use_t=True,
            )
        interval = model.conf_int().loc["inventory_gap_pp"]
        sensitivity_rows.append({
            "outcome": name, "sample": sample_name, "pairs": int(model.nobs), "companies": sample["ticker"].nunique(),
            "association_10pp": model.params["inventory_gap_pp"] * 10,
            "ci_lower_10pp": interval.iloc[0] * 10, "ci_upper_10pp": interval.iloc[1] * 10,
            "r_squared": model.rsquared,
        })
regression_sensitivity = pd.DataFrame(sensitivity_rows)
display(regression_sensitivity.round(4))

          outcome                         sample  pairs  companies  \
0    Gross margin                      All pairs    401         36   
1    Gross margin       Unchanged period lengths    201         36   
2    Gross margin           Common outcome pairs    399         36   
3    Gross margin  Central 98% of inventory gaps    393         36   
4  OCF-to-revenue                      All pairs    490         42   
5  OCF-to-revenue       Unchanged period lengths    235         42   
6  OCF-to-revenue           Common outcome pairs    399         36   
7  OCF-to-revenue  Central 98% of inventory gaps    480         42   

   association_10pp  ci_lower_10pp  ci_upper_10pp  r_squared  
0           -0.1663        -0.3491         0.0164     0.9630  
1           -0.0216        -0.2274         0.1842     0.9535  
2           -0.1688        -0.3513         0.0137     0.9629  
3           -0.1889        -0.4119         0.0341     0.9639  
4            0.1955        -0.1032         0.4943     

In [82]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, name in zip(axes, regression_tables):
    chart = model_results.loc[model_results["outcome"].eq(name) & model_results["model"].str.startswith("Next level")]
    positions = list(range(len(chart)))
    distances = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], positions, xerr=distances, fmt="o", capsize=4, color="tab:blue")
    ax.set_yticks(positions, chart["model"])
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(f"{name}: {len(regression_tables[name])} pairs")
    ax.set_xlabel("Next-year ratio association per 10 pp larger inventory gap (pp)")
    ax.set_ylabel("Model specification")
first_end = min(table["period_end"].min() for table in regression_tables.values()).date()
last_end = max(table["period_end"].max() for table in regression_tables.values()).date()
fig.text(0.5, 0.01, f"Source: Module 9 SEC-based pairs | Current ends: {first_end} to {last_end}\nFirm-clustered 95% coefficient intervals | Retrospective, provisional associations", ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_baseline_control_coefficients.png", dpi=180, bbox_inches="tight")
plt.show()

<Figure size 1100x700 with 2 Axes>

In [83]:
case_union = pd.concat(regression_tables.values(), ignore_index=True).drop_duplicates(["ticker", "period_end"])
high_cases = case_union.sort_values(["inventory_sales_gap", "ticker", "period_end"], ascending=[False, True, True]).drop_duplicates("subgroup").copy()
high_cases["selection_reason"] = "Largest inventory gap within subgroup"
positive = case_union.loc[case_union["inventory_sales_gap"].gt(0)]
distance = (positive["inventory_sales_gap"] - positive["inventory_sales_gap"].median()).abs()
typical_case = positive.loc[[distance.idxmin()]].copy()
typical_case["selection_reason"] = "Closest to the median positive inventory gap"
negative_case = case_union.sort_values(["inventory_sales_gap", "ticker", "period_end"]).head(1).copy()
negative_case["selection_reason"] = "Most negative inventory gap"
cases = pd.concat([high_cases, typical_case, negative_case], ignore_index=True).drop_duplicates(["ticker", "period_end"])
case_columns = [
    "ticker", "subgroup", "period_end", "next_period_end", "accn", "next_accn", "selection_reason",
    "revenue", "revenue_for_analysis", "revenue_growth", "inventory", "inventory_growth", "inventory_gap_pp",
    "gross_margin", "next_gross_margin", "next_gross_margin_change", "ocf_margin", "next_ocf_margin", "next_ocf_margin_change",
    "gm_pair_usable", "ocf_pair_usable", "derived_fields", "revenue_basis_adjusted",
]
cases = cases[case_columns].copy().reset_index(drop=True)
cases["cik10"] = cases["ticker"].map(company_review.set_index("ticker")["cik10"])
for index, row in cases.iterrows():
    for accession_column, url_column in [("accn", "current_filing_url"), ("next_accn", "next_filing_url")]:
        cases.at[index, url_column] = f"https://www.sec.gov/Archives/edgar/data/{int(row['cik10'])}/{row[accession_column].replace('-', '')}/"
cases["review_status"] = "pending"
for column in ["units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "suggested_action", "current_evidence_url", "next_evidence_url", "review_notes"]:
    cases[column] = ""
print("Selected filing review cases:", len(cases))
display(cases[["ticker", "period_end", "next_period_end", "inventory_gap_pp", "selection_reason", "review_status"]])

Selected filing review cases: 7


  ticker period_end next_period_end  inventory_gap_pp  \
0    AKA 2021-12-31      2022-12-31         89.169269   
1   SPWH 2022-01-29      2023-01-28         55.053967   
2   ARHS 2021-12-31      2022-12-31         35.819959   
3   ULTA 2013-02-02      2014-02-01         22.606853   
4    KSS 2013-02-02      2014-02-01         14.635555   
5   URBN 2011-01-31      2012-01-31          5.979765   
6   SPWH 2021-01-30      2022-01-29        -75.422998   

                               selection_reason review_status  
0         Largest inventory gap within subgroup       pending  
1         Largest inventory gap within subgroup       pending  
2         Largest inventory gap within subgroup       pending  
3         Largest inventory gap within subgroup       pending  
4         Largest inventory gap within subgroup       pending  
5  Closest to the median positive inventory gap       pending  
6                   Most negative inventory gap       pending  

In [84]:
outputs = {
    "expanded_baseline_regression_results.csv": model_results,
    "expanded_baseline_identity_check.csv": identity_check,
    "expanded_adjusted_regression_sensitivity.csv": regression_sensitivity,
    "expanded_module11_case_queue.csv": cases,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
review_path = PROCESSED_DIR / "expanded_module11_case_review.csv"
if not review_path.exists():
    cases.to_csv(review_path, index=False, date_format="%Y-%m-%d")
else:
    print("Existing manual case review retained:", review_path.name)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module10_metadata": module10_metadata,
    "statsmodels_version": statsmodels.__version__, "sample_status": "provisional", "model_specs": model_specs,
    "model_weighting": "OLS on company-year pairs; not company-equal weighting",
    "covariance_rule": "Firm-clustered CRV1 with small-sample and cluster-df corrections; t-based 95% coefficient intervals",
    "year_rule": "Categorical calendar year of target period end; not fully aligned operating periods",
    "unit": "Ratio levels in percent; growth gaps and changes in percentage points; coefficients scaled to a 10 pp gap",
    "main_model_rows": len(model_results), "sensitivity_rows": len(regression_sensitivity), "case_count": len(cases),
    "case_selection": "Subgroup maxima, median positive-gap representative and minimum gap; use current signals only",
    "manual_review_decisions_applied": module10_metadata.get("module9_metadata", {}).get("manual_review_decisions_applied", False), "confidence_intervals_reported": True, "p_values_reported": False,
    "accounting_scope_verified": False, "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False,
}
(PROCESSED_DIR / "expanded_module11_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 11 tables saved in:", PROCESSED_DIR)
print("Module 11 figure saved in:", FIGURE_DIR)

Existing manual case review retained: expanded_module11_case_review.csv
Module 11 tables saved in: [project root]/work/data/processed/module12_reviewed
Module 11 figure saved in: [project root]/work/data/processed/module12_reviewed/figures


## Module 13: Final Research Report and Project Package

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import base64
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
REPORT_DATE = "2026-10-11"
DATA_DIR = PROJECT_ROOT / "work" / "data" / "processed" / "module12_reviewed"
REVIEW_DIR = DATA_DIR.parent / "module12_manual_review"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Report inputs:", DATA_DIR)
print("Report outputs:", REPORT_DIR)


Report inputs: [project root]/work/data/processed/module12_reviewed
Report outputs: [project root]/outputs/retail_inventory_final


In [2]:
input_files = {
    "annual": "expanded_module12_annual_audit.csv",
    "gm_pairs": "expanded_gm_next_year_pairs.csv",
    "ocf_pairs": "expanded_ocf_next_year_pairs.csv",
    "readiness": "expanded_module12_report_readiness.csv",
    "pooled": "expanded_inventory_group_summary.csv",
    "balanced": "expanded_company_balanced_summary.csv",
    "models": "expanded_baseline_regression_results.csv",
    "sensitivity": "expanded_adjusted_regression_sensitivity.csv",
    "cases": "expanded_module12_case_results.csv",
    "companies": "candidate_scope_review.csv",
    "applied": "expanded_module12_applied_values.csv",
}
tables = {}
source_paths = []
for name, filename in input_files.items():
    path = DATA_DIR / filename
    tables[name] = pd.read_csv(path, keep_default_na=False)
    source_paths.append(path)
metadata = {}
for module in [12, 10, 11]:
    path = DATA_DIR / f"expanded_module{module}_metadata.json"
    metadata[module] = json.loads(path.read_text(encoding="utf-8"))
    source_paths.append(path)
assert metadata[10]["module9_metadata"] == metadata[12], "Run Module 10 after the latest Module 12."
assert metadata[11]["module10_metadata"] == metadata[10], "Run Module 11 after the latest Module 10."
for filename, table_name, keys, fields in [
    ("company_review.csv", "companies", ["ticker"], ["eligibility_review_status", "include_decision", "eligibility_evidence_url", "eligibility_notes"]),
    ("annual_scope_review.csv", "annual", ["ticker", "period_end"], ["scope_status", "scope_evidence_url", "scope_notes"]),
    ("case_review.csv", "cases", ["ticker", "period_end"], ["review_status", "suggested_action", "units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "current_evidence_url", "next_evidence_url", "review_notes"]),
]:
    path = REVIEW_DIR / filename
    manual = pd.read_csv(path, keep_default_na=False)
    left = manual[keys + fields].sort_values(keys).reset_index(drop=True)
    right = tables[table_name][keys + fields].sort_values(keys).reset_index(drop=True)
    assert left.equals(right), "New manual decisions need Module 12, then Modules 10 and 11: " + filename
    source_paths.append(path)
metric_path = REVIEW_DIR / "metric_review.csv"
metric_review = pd.read_csv(metric_path, keep_default_na=False)
approved = metric_review.loc[metric_review["review_status"].eq("approved")]
assert len(approved) == len(tables["applied"]), "Apply new metric approvals in Module 12 before reporting."
source_paths.append(metric_path)
for name, outcome in [("gm_pairs", "Gross margin"), ("ocf_pairs", "OCF-to-revenue")]:
    counts = tables["models"].loc[tables["models"]["outcome"].eq(outcome), "pairs"]
    assert counts.eq(len(tables[name])).all(), "Model sample counts do not match the applied pairs."
print("Applied decisions and saved analysis versions agree.")


Applied decisions and saved analysis versions agree.


In [3]:
sample_rows = []
for name, outcome in [("gm_pairs", "Gross margin"), ("ocf_pairs", "OCF-to-revenue")]:
    table = tables[name]
    sample_rows.append({
        "Outcome": outcome, "Pairs": len(table), "Companies": table["ticker"].nunique(),
        "Reviewed pairs": int(table["review_complete_pair"].sum()),
        "Current end from": table["period_end"].min(), "Current end to": table["period_end"].max(),
        "Latest target end": table["next_period_end"].max(),
        "Before target start": int(table["available_before_target_start"].sum()),
        "Different lengths": int(table["pair_period_length_changed"].sum()),
    })
sample_summary = pd.DataFrame(sample_rows)
common_keys = tables["gm_pairs"][["ticker", "period_end"]].merge(
    tables["ocf_pairs"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one"
)
reviewed_cases = tables["cases"].loc[tables["cases"]["review_status"].eq("reviewed")].copy()
pending_cases = tables["cases"].loc[tables["cases"]["review_status"].eq("pending")].copy()
review_summary = pd.DataFrame([
    {"Review item": "Company eligibility", "Reviewed": int(tables["companies"]["eligibility_review_status"].eq("reviewed").sum()), "Total": len(tables["companies"])},
    {"Review item": "Annual scope", "Reviewed": int(tables["annual"]["scope_status"].eq("reviewed").sum()), "Total": len(tables["annual"])},
    {"Review item": "Selected cases", "Reviewed": len(reviewed_cases), "Total": len(tables["cases"])},
])
review_summary["Pending"] = [
    int(tables["companies"]["eligibility_review_status"].eq("pending").sum()),
    int(tables["annual"]["scope_status"].eq("pending").sum()), len(pending_cases),
]
missing_cost = int(pd.to_numeric(tables["annual"]["cost_of_sales"], errors="coerce").isna().sum())
missing_gp = int(pd.to_numeric(tables["annual"]["gross_profit"], errors="coerce").isna().sum())
display(sample_summary)
display(review_summary)
print("Common outcome pairs:", len(common_keys))
print("Remaining missing cost / gross profit:", missing_cost, "/", missing_gp)


          Outcome  Pairs  Companies  Reviewed pairs Current end from  \
0    Gross margin    401         36               3       2010-01-29   
1  OCF-to-revenue    490         42               3       2010-01-29   

  Current end to Latest target end  Before target start  Different lengths  
0     2025-08-02        2026-08-01                    0                200  
1     2025-08-02        2026-08-01                    0                255  

           Review item  Reviewed  Total  Pending
0  Company eligibility         2     42       40
1         Annual scope         7    656      649
2       Selected cases         3      7        4

Common outcome pairs: 399
Remaining missing cost / gross profit: 42 / 108


In [4]:
main_results = tables["models"].loc[tables["models"]["model"].eq("Next level: company/year")].copy()
assert len(main_results) == 2, "Expected one adjusted model for each outcome."
main_table = main_results[["outcome", "pairs", "companies", "association_10pp", "ci_lower_10pp", "ci_upper_10pp"]].rename(columns={
    "outcome": "Outcome", "pairs": "Pairs", "companies": "Companies", "association_10pp": "Association per 10pp (pp)",
    "ci_lower_10pp": "95% lower (pp)", "ci_upper_10pp": "95% upper (pp)",
})
pooled_table = tables["pooled"][["outcome", "inventory_group", "pairs", "mean_change_pp", "median_change_pp", "decline_rate"]].copy()
pooled_table["decline_rate"] *= 100
pooled_table.columns = ["Outcome", "Inventory group", "Pairs", "Mean change (pp)", "Median change (pp)", "Decline rate (%)"]
balanced_table = tables["balanced"].rename(columns={
    "outcome": "Outcome", "companies": "Eligible companies", "mean_company_difference_pp": "Mean company difference (pp)",
    "median_company_difference_pp": "Median company difference (pp)", "share_negative": "Share negative (%)",
}).copy()
balanced_table["Share negative (%)"] *= 100
caption = f"Source: Modules 10-12; current ends {sample_summary['Current end from'].min()} to {sample_summary['Current end to'].max()}; partial filing review."
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = tables["models"].loc[tables["models"]["outcome"].eq(outcome) & tables["models"]["model"].str.startswith("Next level")]
    error = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], range(len(chart)), xerr=error, fmt="o", color="#245a73", capsize=4)
    ax.set_yticks(range(len(chart)), chart["model"].str.replace("Next level: ", "", regex=False))
    ax.set_xlabel("Estimated association per 10pp inventory gap (pp)")
    ax.set_ylabel("Model specification")
    ax.set_title(outcome)
    ax.axvline(0, color="gray", linewidth=1)
fig.suptitle("Next-year ratios: estimates and firm-clustered 95% intervals")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = tables["pooled"].loc[tables["pooled"]["outcome"].eq(outcome)]
    ax.bar(chart["inventory_group"], chart["mean_change_pp"], color=["#a9b2b8", "#245a73"])
    ax.set_xlabel("Current inventory growth minus revenue growth")
    ax.set_ylabel("Mean next-year ratio change (pp)")
    ax.set_title(outcome)
    ax.axhline(0, color="gray", linewidth=1)
fig.suptitle("Descriptive group means: annual pairs receive equal weight")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "group_means.png", dpi=160, bbox_inches="tight")
plt.show()
display(main_table.round(4))


<Figure size 1200x500 with 2 Axes>

<Figure size 1100x400 with 2 Axes>

          Outcome  Pairs  Companies  Association per 10pp (pp)  \
4    Gross margin    401         36                    -0.1663   
9  OCF-to-revenue    490         42                     0.1955   

   95% lower (pp)  95% upper (pp)  
4         -0.3491          0.0164  
9         -0.1032          0.4943  

In [5]:
sensitivity = tables["sensitivity"].copy()
sensitivity["zero_in_interval"] = sensitivity["ci_lower_10pp"].le(0) & sensitivity["ci_upper_10pp"].ge(0)
sensitivity_table = sensitivity[["outcome", "sample", "pairs", "companies", "association_10pp", "ci_lower_10pp", "ci_upper_10pp"]].copy()
sensitivity_table.columns = ["Outcome", "Sample rule", "Pairs", "Companies", "Association per 10pp (pp)", "95% lower (pp)", "95% upper (pp)"]
sensitivity_notes = []
for outcome, group in sensitivity.groupby("outcome", sort=False):
    count = int(group["zero_in_interval"].sum())
    sensitivity_notes.append(f"{outcome}: {count} of {len(group)} adjusted sensitivity intervals include zero.")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, outcome in zip(axes, ["Gross margin", "OCF-to-revenue"]):
    chart = sensitivity.loc[sensitivity["outcome"].eq(outcome)]
    labels = chart["sample"] + " (n=" + chart["pairs"].astype(str) + ")"
    error = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], range(len(chart)), xerr=error, fmt="o", color="#245a73", capsize=4)
    ax.set_yticks(range(len(chart)), labels)
    ax.set_xlabel("Adjusted association per 10pp inventory gap (pp)")
    ax.set_ylabel("Sample rule and pair count")
    ax.set_title(outcome)
    ax.axvline(0, color="gray", linewidth=1)
fig.suptitle("Adjusted sensitivity estimates and firm-clustered 95% intervals")
fig.text(0.5, 0.015, caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 0.92])
fig.savefig(FIGURE_DIR / "sensitivity.png", dpi=160, bbox_inches="tight")
plt.show()
display(sensitivity_table.round(4))
print(" ".join(sensitivity_notes))


<Figure size 1300x500 with 2 Axes>

          Outcome                    Sample rule  Pairs  Companies  \
0    Gross margin                      All pairs    401         36   
1    Gross margin       Unchanged period lengths    201         36   
2    Gross margin           Common outcome pairs    399         36   
3    Gross margin  Central 98% of inventory gaps    393         36   
4  OCF-to-revenue                      All pairs    490         42   
5  OCF-to-revenue       Unchanged period lengths    235         42   
6  OCF-to-revenue           Common outcome pairs    399         36   
7  OCF-to-revenue  Central 98% of inventory gaps    480         42   

   Association per 10pp (pp)  95% lower (pp)  95% upper (pp)  
0                    -0.1663         -0.3491          0.0164  
1                    -0.0216         -0.2274          0.1842  
2                    -0.1688         -0.3513          0.0137  
3                    -0.1889         -0.4119          0.0341  
4                     0.1955         -0.1032          

Gross margin: 4 of 4 adjusted sensitivity intervals include zero. OCF-to-revenue: 4 of 4 adjusted sensitivity intervals include zero.


In [6]:
reviewed_keys = pd.concat([
    tables["gm_pairs"][["ticker", "period_end", "review_complete_pair"]],
    tables["ocf_pairs"][["ticker", "period_end", "review_complete_pair"]],
]).drop_duplicates(["ticker", "period_end"])
reviewed_cases = reviewed_cases.merge(reviewed_keys, on=["ticker", "period_end"], how="left", validate="one_to_one")
case_table = reviewed_cases[["ticker", "period_end", "inventory_gap_pp", "next_gross_margin_change", "next_ocf_margin_change", "suggested_action", "review_complete_pair"]].copy()
for column in ["next_gross_margin_change", "next_ocf_margin_change"]:
    case_table[column] = pd.to_numeric(case_table[column], errors="coerce") * 100
case_table.columns = ["Ticker", "Current period end", "Inventory gap (pp)", "Next GM change (pp)", "Next OCF/revenue change (pp)", "Decision", "Three-year review passed"]
pending_table = pending_cases[["ticker", "period_end", "selection_reason"]].copy()
pending_table.columns = ["Ticker", "Current period end", "Selection reason"]
display(case_table.round(4))
display(pending_table)


  Ticker Current period end  Inventory gap (pp)  Next GM change (pp)  \
0   SPWH         2022-01-29             55.0540               0.3310   
1   URBN         2011-01-31              5.9798              -6.4004   
2   SPWH         2021-01-30            -75.4230              -0.2642   

   Next OCF/revenue change (pp) Decision  Three-year review passed  
0                        4.7795   retain                      True  
1                       -5.5069   retain                      True  
2                      -17.8859   retain                      True  

  Ticker Current period end                       Selection reason
0    AKA         2021-12-31  Largest inventory gap within subgroup
2   ARHS         2021-12-31  Largest inventory gap within subgroup
3   ULTA         2013-02-02  Largest inventory gap within subgroup
4    KSS         2013-02-02  Largest inventory gap within subgroup

In [7]:
adjusted_sentences = []
for _, row in main_results.iterrows():
    conclusion = "The 95% interval includes zero; the estimate remains imprecise."
    if row["ci_lower_10pp"] > 0 or row["ci_upper_10pp"] < 0:
        conclusion = "The 95% interval excludes zero in this specification; robustness and design limits still matter."
    adjusted_sentences.append(
        f"{row['outcome']}: a 10 percentage-point inventory-growth gap is associated with {row['association_10pp']:+.4f} percentage points in the next-year ratio, conditional on the controls (95% interval {row['ci_lower_10pp']:+.4f} to {row['ci_upper_10pp']:+.4f}). {conclusion}"
    )
all_intervals_include_zero = bool(sensitivity["zero_in_interval"].all())
if all_intervals_include_zero:
    final_conclusion = "The adjusted specifications examined do not provide sufficiently precise evidence of a stable association. This does not establish that the true association is zero or that inventory information has no research value."
else:
    final_conclusion = "The specifications provide mixed evidence that must be assessed using the full set of estimates and intervals, rather than a selected significant result."
company_count = len(tables["companies"])
annual_count = len(tables["annual"])
company_review_count = int(review_summary.loc[review_summary["Review item"].eq("Company eligibility"), "Reviewed"].iloc[0])
annual_review_count = int(review_summary.loc[review_summary["Review item"].eq("Annual scope"), "Reviewed"].iloc[0])
pending_names = ", ".join(pending_cases["ticker"].drop_duplicates()) or "None"
timing_min = int(min(tables[name]["days_after_target_start"].min() for name in ["gm_pairs", "ocf_pairs"]))
timing_max = int(max(tables[name]["days_after_target_start"].max() for name in ["gm_pairs", "ocf_pairs"]))
report_sections = {
    "Executive Summary": f"This project studies inventory growth relative to revenue growth and subsequent operating performance among {company_count} manually selected U.S.-listed merchandise retailers and consumer brands. It uses {annual_count} annual records, separate gross-margin and OCF-to-revenue research samples, and {len(common_keys)} common outcome pairs.\n\n" + "\n\n".join(adjusted_sentences) + f"\n\nFiling review covers {company_review_count} company eligibility decisions, {annual_review_count} annual rows and {len(reviewed_cases)} of {len(tables['cases'])} selected cases. The statistical sample remains provisional. " + final_conclusion,
    "Research Question and Economic Rationale": "Does inventory growth exceeding revenue growth in year t relate to changes in gross margin and operating cash flow divided by revenue in year t+1?\n\nInventory-growth gap = (Inventory_t / Inventory_(t-1) - 1) - (Revenue_t / Revenue_(t-1) - 1). Growth differences are expressed in percentage points. Gross margin = reported gross profit / analysis revenue. OCF-to-revenue = operating cash flow / analysis revenue. Change outcomes equal the next-year ratio minus the current-year ratio.\n\nInventory accumulation may signal slower demand, subsequent markdowns or working-capital pressure. Expansion, acquisitions, deliberate replenishment and product mix can produce similar signals with different outcomes. A positive gap is a research flag whose interpretation requires context.",
    "Sample and Review Coverage": f"The {company_count}-company candidate universe was manually assembled across five researcher-defined business groups: apparel/footwear/off-price, department stores, home/furniture/home improvement, sporting/electronics/other merchandise, and beauty/personal care. It is not a census, random sample or certified GICS classification. U.S. listing does not imply exclusively U.S. operations or pure retail revenue.\n\nCompany identity is mapped to SEC CIK. Company Facts JSON snapshots supply standard USD facts. Annual revenue periods in 10-K/10-K/A filings anchor the panel; the extraction matches periods and accession numbers, distinguishing flow measures from year-end balances. The earliest visible filing in the cached facts is an extraction rule, not historical point-in-time certification.\n\nOutcome-specific usable annual coverage of at least five years, continuity, numeric checks and documented scope restrictions define an unbalanced panel. The pipeline preserves source tags, limited flagged derivations, original amounts and separate analysis values. Module 9 reconciled identified reporting differences, including DDS net-sales denominators and separately presented gross-profit charges. Other scope and classification differences remain subject to review.\n\nReviewed pairs denote recorded company eligibility plus prior/current/next annual-scope checks. This is a research review status, not an independent audit. No new metric values were assumed for pending decisions. Applied metric values: {len(tables['applied'])}.",
    "Methods": "The project compares positive and nonpositive inventory gaps, examines correlations and company-level patterns, and estimates OLS models on company-year pairs. Company-balanced descriptive comparisons require at least three observations in each gap group for a company.\n\nThe main adjusted model explains the next-year ratio using inventory-growth gap, the current ratio, revenue growth, log total assets, company fixed effects and target-end-calendar-year fixed effects. Inference uses firm-clustered CRV1 covariance with small-sample corrections and t-based 95% intervals. The estimates are scaled to a 10 percentage-point inventory gap.\n\nCurrent-ratio controls address starting-level differences and mean-reversion patterns. Baseline-controlled change and next-level specifications have an algebraic coefficient relationship; they are not independent replications. Company/year controls reduce some confounding but do not identify causal effects. Calendar-year effects do not fully align fiscal operating periods.\n\nSensitivity rules examine all pairs, periods without length changes exceeding three days, common outcome pairs, and central 98% of outcome-specific inventory gaps. Trimming uses the signal, not future outcomes. R-squared measures in-sample fit and is not predictive accuracy.",
    "Descriptive Results": "The tables distinguish pooled annual-pair summaries from company-equal comparisons. Mean changes, median changes and decline frequencies capture different distribution features. A positive group difference means a more positive next-year change in Gap > 0 relative to Gap <= 0.\n\nThese descriptive differences are not causal effects and do not incorporate the full control set. A descriptive positive cash-flow association can coexist with an imprecise adjusted estimate; the full model comparison should be consulted.",
    "Adjusted Regression Results": "\n\n".join(adjusted_sentences) + "\n\nThe fully adjusted estimates should be interpreted alongside the alternative control specifications in the figure. Any conclusion is conditional on the provisional sample, measured variables and inference assumptions.",
    "Sensitivity Analysis": " ".join(sensitivity_notes) + "\n\n" + final_conclusion + " Smaller subsets also change sample composition and precision. The four rules are sensitivity checks, not independent experiments.",
    "Filing-reviewed Cases": "Cases were selected from available research pairs using subgroup maximum current gaps, a representative near the median positive gap, and the minimum gap. Selection did not use subsequent outcome direction, although entry into the research-pair pool depends on outcome-data availability. These purposefully selected cases cannot estimate population frequencies.\n\nThe table and evidence narratives include only cases marked reviewed. Management explanations, statement facts and research interpretation should be distinguished. Retain means no additional exclusion was justified by the recorded review; it does not certify causality.",
    "Research Limits and Remaining Work": f"The manually selected present-day candidate list can create selection and survivorship bias. Unbalanced history and outcome-dependent missingness affect generalization. Cost and gross-profit classifications differ across firms; fixed effects do not remove every accounting-comparability problem. Missing annual cost fields: {missing_cost}; missing annual gross-profit fields: {missing_gp}.\n\nFiling review is partial. Pending selected cases: {pending_names}. Most annual rows and company histories have not completed scope review. Recorded derivations and standard-tag selections do not constitute complete historical verification.\n\nCurrent filings arrive {timing_min}-{timing_max} days after the target year begins in these retained samples. The design is retrospective and may overlap the target operating period. It is not a before-year-start forecast. Latest cached comparatives and incomplete historical information sets prevent point-in-time certification.\n\nThe project has no out-of-sample prediction validation, stock-return backtest or causal identification. Small business groups and a limited number of firm clusters constrain inference. Wide intervals can reflect limited precision; they do not prove absence of an effect.\n\nFurther work can complete pending case reviews, prioritize material missing fields, assess same-store sales and store expansion, and define an explicit filing-date-based forecast horizon before attempting predictive validation. These extensions are not claimed as completed in this version.",
    "Practical Use and Conclusion": "The practical contribution is a reproducible fundamental-research workflow connecting a transparent inventory signal to accounting checks, subsequent operating ratios and cited filing evidence. It can help prioritize companies for deeper investigation of demand, replenishment, expansion, vendor terms and cash-flow composition.\n\n" + final_conclusion + " Inventory-growth gaps should be interpreted with business context and a fuller information set. This version completes the analysis/report deliverable with explicit remaining-review limits; it does not establish a trading rule or validated prediction model.",
    "Sources and Reproduction": f"Primary inputs are cached SEC Company Facts and annual filings. Amounts use USD; growth rates and ratios are stored as decimals and displayed as percent or percentage points as labeled. Applied-review snapshot: {metadata[12]['processed_at_utc']}. Statistical results were regenerated from that snapshot through Modules 10 and 11.\n\nThe companion source_manifest.json identifies every input file with a SHA-256 hash. Reviewed-case links below point to the original filings; detailed accounting and annual-scope evidence remain in the input audit tables. README.md describes execution order and deliverable structure. The report date labels this deliverable, not the underlying financial snapshot.",
}
references = [
    ("SEC: EDGAR Application Programming Interfaces", "https://www.sec.gov/search-filings/edgar-application-programming-interfaces"),
    ("Gaur, Fisher, and Raman (2005). An Econometric Analysis of Inventory Turnover Performance in Retail Services. Management Science, 51(2), 181-194.", "https://pubsonline.informs.org/doi/10.1287/mnsc.1040.0298"),
]
report_sections["Research Context and References"] = "Gaur, Fisher, and Raman (2005) examine retail inventory turnover in relation to gross margin, capital intensity and sales surprise. Their study provides background for considering multiple operating factors when interpreting inventory metrics. The present project evaluates subsequent operating ratios using a different inventory-growth-gap measure.\n\nSEC documentation describes the Company Facts API as aggregating standard-taxonomy facts applying to the entire filing entity. Custom-tag disclosures can therefore require direct filing review. The project records unresolved fields and selected filing evidence."
report_tables = {
    "Sample and Review Coverage": [sample_summary, review_summary],
    "Descriptive Results": [pooled_table.round(4), balanced_table.round(4)],
    "Adjusted Regression Results": [main_table.round(4)],
    "Sensitivity Analysis": [sensitivity_table.round(4)],
    "Filing-reviewed Cases": [case_table.round(4)],
    "Research Limits and Remaining Work": [pending_table],
}
section_images = {"Descriptive Results": "group_means.png", "Adjusted Regression Results": "model_comparison.png", "Sensitivity Analysis": "sensitivity.png"}
report_title = "Inventory Growth and Subsequent Operating Performance"
status_line = f"Final report | Historical associations | Partial filing review | Report date: {REPORT_DATE}"
markdown_parts = [f"# {report_title}\n\n{status_line}\n\nSelected U.S.-listed merchandise retailers and consumer brands.\n\n"]
html_parts = [f"<header><p class='eyebrow'>RETAIL FUNDAMENTALS RESEARCH</p><h1>{escape(report_title)}</h1><p>{escape(status_line)}</p><p>Selected U.S.-listed merchandise retailers and consumer brands.</p></header>"]
for title, text in report_sections.items():
    text = text.replace("\\n", "\n")
    markdown_parts.append(f"## {title}\n\n{text}\n\n")
    html_parts.append(f"<section><h2>{escape(title)}</h2>")
    for paragraph in text.split("\n\n"):
        html_parts.append("<p>" + escape(paragraph) + "</p>")
    for table in report_tables.get(title, []):
        if table.empty:
            continue
        markdown_parts.append("| " + " | ".join(table.columns) + " |\n")
        markdown_parts.append("| " + " | ".join(["---"] * len(table.columns)) + " |\n")
        for _, row in table.fillna("").iterrows():
            markdown_parts.append("| " + " | ".join(str(value).replace("|", "/").replace("\n", " ") for value in row) + " |\n")
        markdown_parts.append("\n")
        html_parts.append("<div class='table-wrap'>" + table.to_html(index=False, border=0, escape=True) + "</div>")
    if title in section_images:
        filename = section_images[title]
        encoded = base64.b64encode((FIGURE_DIR / filename).read_bytes()).decode("ascii")
        markdown_parts.append(f"![{title}](figures/{filename})\n\n")
        html_parts.append(f"<figure><img alt='{escape(title)}' src='data:image/png;base64,{encoded}'><figcaption>{escape(caption)}</figcaption></figure>")
    if title == "Filing-reviewed Cases":
        for _, row in reviewed_cases.iterrows():
            label = f"{row['ticker']} | Current period end {row['period_end']} | Next period end {row['next_period_end']}"
            markdown_parts.append(f"### {label}\n\n")
            html_parts.append(f"<article class='case'><h3>{escape(label)}</h3>")
            for field, label_text in [("units_and_scope_notes", "Units and scope"), ("revenue_and_inventory_drivers", "Current drivers"), ("next_margin_and_ocf_drivers", "Next-year drivers"), ("review_notes", "Review interpretation")]:
                note = str(row[field])
                markdown_parts.append(f"**{label_text}:** {note}\n\n")
                html_parts.append(f"<p><strong>{escape(label_text)}:</strong> {escape(note)}</p>")
            for field, label_text in [("current_evidence_url", "Current annual filing"), ("next_evidence_url", "Next annual filing")]:
                url = str(row[field])
                markdown_parts.append(f"[{label_text}]({url})\n\n")
                html_parts.append(f"<p><a href='{escape(url, quote=True)}'>{escape(label_text)}</a></p>")
            html_parts.append("</article>")
    if title == "Research Context and References":
        for label, url in references:
            markdown_parts.append(f"- [{label}]({url})\n\n")
            html_parts.append(f"<p><a href='{escape(url, quote=True)}'>{escape(label)}</a></p>")
    html_parts.append("</section>")
markdown_report = "".join(markdown_parts)
style = "body{font-family:Arial,sans-serif;color:#22313a;background:#fff;margin:0;line-height:1.65}main{max-width:1080px;margin:auto;padding:40px 28px 70px}header{border-bottom:2px solid #245a73;padding-bottom:24px}h1{font-size:30px;line-height:1.2;max-width:820px}h2{font-size:21px;margin:35px 0 12px}h3{font-size:17px}p{margin:12px 0}.eyebrow{font-size:12px;letter-spacing:1px;color:#245a73}.table-wrap{overflow-x:auto;margin:20px 0}table{width:100%;border-collapse:collapse;font-size:13px}th,td{text-align:left;vertical-align:top;padding:9px 10px;border-bottom:1px solid #d8dfe3}th{background:#eef3f5}img{width:100%;height:auto}figure{margin:24px 0}figcaption{font-size:11px;color:#52616b}.case{border-top:1px solid #d8dfe3;margin-top:25px;padding-top:12px}a{color:#245a73}@media print{main{padding:0}h2,h3{break-after:avoid}figure{break-inside:avoid}.table-wrap{overflow:visible}table{font-size:10px}a{color:inherit}}"
html_report = f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>{escape(report_title)}</title><style>{style}</style></head><body><main>{''.join(html_parts)}</main></body></html>"
(REPORT_DIR / "Retail_Inventory_Research_Report.md").write_text(markdown_report, encoding="utf-8")
(REPORT_DIR / "Retail_Inventory_Research_Report.html").write_text(html_report, encoding="utf-8")
source_manifest = {"report_date": REPORT_DATE, "generated_at_utc": datetime.now(timezone.utc).isoformat(), "sample_status": metadata[12]["sample_status"], "manual_review_decisions_applied": metadata[12]["manual_review_decisions_applied"], "accounting_scope_verified": metadata[12]["accounting_scope_verified"], "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False, "references": references, "sources": []}
for path in source_paths:
    source_manifest["sources"].append({"path": str(path.relative_to(PROJECT_ROOT)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
(REPORT_DIR / "source_manifest.json").write_text(json.dumps(source_manifest, indent=2), encoding="utf-8")
print("English Markdown and standalone HTML reports saved.")


English Markdown and standalone HTML reports saved.


In [8]:
readme = f"""# Retail Inventory Research: Final Deliverable

Report date: {REPORT_DATE}
Scope: retrospective operating-performance associations with partial filing review.

## Files
- Retail_Inventory_Research_Report.html: standalone report with embedded figures.
- Retail_Inventory_Research_Report.md: editable report; keep the figures directory beside it.
- figures/: model comparison, descriptive group means, and adjusted sensitivity intervals.
- source_manifest.json: relative input paths and SHA-256 hashes for this report version.

## Reproduction
Use the existing project root and cached SEC data. Required libraries are pandas, matplotlib, requests, numpy, statsmodels, and IPython.
The completed notebook preserves the earlier modules and their outputs, and appends the executed report module.
Its established-project execution order is Modules 1-9, 12, 10, 11, 13. Module 12 uses the initial Module 11 case file already present in the project.
For a fresh project, first follow the original Guides through Module 11 to create inputs and the initial case file; then run Module 12, Module 10, Module 11, and Module 13.
Set PROJECT_ROOT for your machine. Update REPORT_DATE when generating a new report version.
After changing manual review CSVs, run Module 12 to apply decisions, Modules 10 and 11 to refresh analyses, and all eight Module 13 cells to rebuild the report.
Module 13 alone reads saved inputs; it does not acquire data, audit filings, or refit models.
The version checks intentionally stop reporting if saved analyses or manual decisions are stale.

## Completed and Remaining Work
The analysis/report deliverable is complete for this snapshot. Full historical filing review, prediction validation, and causal identification are not complete.
Reviewed cases: {len(reviewed_cases)} of {len(tables['cases'])}. Pending selected cases: {pending_names}.
The report records pending work and does not fabricate supporting evidence or fill unapproved values.
"""
(REPORT_DIR / "README.md").write_text(readme, encoding="utf-8")
deliverables = ["Retail_Inventory_Research_Report.md", "Retail_Inventory_Research_Report.html", "source_manifest.json", "README.md", "figures/model_comparison.png", "figures/group_means.png", "figures/sensitivity.png"]
for filename in deliverables:
    path = REPORT_DIR / filename
    assert path.exists() and path.stat().st_size > 0, "Missing deliverable: " + filename
    print("Saved:", path)
display(Markdown("## Executive Summary\n\n" + report_sections["Executive Summary"].replace("\\n", "\n")))
display(main_table.round(4))
print("Report version complete; remaining review limits are disclosed.")


Saved: [project root]/outputs/retail_inventory_final/Retail_Inventory_Research_Report.md
Saved: [project root]/outputs/retail_inventory_final/Retail_Inventory_Research_Report.html
Saved: [project root]/outputs/retail_inventory_final/source_manifest.json
Saved: [project root]/outputs/retail_inventory_final/README.md
Saved: [project root]/outputs/retail_inventory_final/figures/model_comparison.png
Saved: [project root]/outputs/retail_inventory_final/figures/group_means.png
Saved: [project root]/outputs/retail_inventory_final/figures/sensitivity.png


<IPython.core.display.Markdown object>

          Outcome  Pairs  Companies  Association per 10pp (pp)  \
4    Gross margin    401         36                    -0.1663   
9  OCF-to-revenue    490         42                     0.1955   

   95% lower (pp)  95% upper (pp)  
4         -0.3491          0.0164  
9         -0.1032          0.4943  

Report version complete; remaining review limits are disclosed.
